# A Calibration-Aware, Threshold-Optimised and Explainable Pipeline for Pneumonia Detection from Chest X-Ray Images

This notebook implements a computer-science-oriented, reproducible AI evaluation pipeline for pneumonia detection from chest X-ray images. The pipeline is deliberately framed for Elsevier *Array*: reproducible benchmark execution, probability calibration, threshold-sensitive evaluation, uncertainty-aware diagnostics, explainable AI outputs, and structured reporting.

The notebook uses a pure reproducible evaluation-framework identity. The workflow component is treated only as a Multi-Criteria Decision Score (MCDS): a structured evaluation criterion for comparing calibrated probabilities, threshold behavior, uncertainty, action burden, and safety-oriented review recommendations. Medical interpretation is kept conservative: the system is a benchmarked decision-support pipeline, not an autonomous diagnostic tool.

Upgrade note: the calibration layer now uses a ranking-preserving primary calibration policy. Temperature scaling is retained as the main reported calibrated model because it preserves probability ranking and avoids artificial ROC-AUC degradation. Platt scaling and isotonic regression are kept as reproducible comparison baselines rather than automatically replacing the primary calibrated model.

Main outputs include calibrated metrics, reliability curves, threshold sensitivity tables, uncertainty-stratified performance, MCDS-based structured evaluation, safety-oriented audit tables, external transferability validation, and an output summary suitable for manuscript revision and GitHub reproducibility.



## Final Array submission upgrade summary

This version keeps the notebook as a reproducible AI evaluation framework rather than a medical-deployment manuscript. The remaining calibration-reporting issue has been corrected: ECE is computed with a robust binning function, stored explicitly as `RAW_ECE_REFERENCE` and `FINAL_CAL_ECE`, and recovered safely in the final summary. Temperature scaling remains the manuscript-facing calibration method because it preserves ROC-AUC ranking while providing stable probability reporting. Isotonic regression and Platt scaling remain transparent comparison baselines only.

The MCDS component is retained only as a structured workflow evaluation criterion, not as an autonomous diagnostic layer.


## Maintenance changelog (code-quality pass)

The following corrections and small robustness improvements were applied to the
pipeline. They do not change the scientific framing; items marked *(affects
numbers)* alter computed values and should be re-run before regenerating
manuscript tables.

1. **Binary-entropy normalisation fixed** *(affects numbers)*. `entropy_binary`
   already returns entropy in bits (range `[0, 1]`), but the normalised
   uncertainty was divided by `ln(2)` a second time, inflating it by ~1.44x
   before clipping. This biased the uncertainty term and the safety-feasibility
   gate (`u > 0.45`). The redundant division was removed in `mcds_action_scores`
   (and in the diagnostic-only top-level block).
2. **Final-summary calibration lookup fixed**. The summary referenced
   `calibration_method_comparison_df`, but the in-memory table is named
   `calibration_comparison_df`; the lookup always failed and fell back to the
   CSV. It now reads the in-memory DataFrame, with the CSV fallback retained.
3. **MC-dropout batching**. `mc_dropout_predict` declared `batch_size` but
   pushed the whole input through in one forward pass; it now batches, which is
   numerically equivalent for inference-time dropout and bounds memory use.
4. **Reproducibility hardening**. `PYTHONHASHSEED` is set and
   `keras.utils.set_random_seed` is called (when available) alongside the
   existing NumPy/`random`/TensorFlow seeds.

- Added a merged review pass that preserves the attached code-quality improvements and corrects minor wording in comments.


## 1. Environment, Drive paths, and reproducibility settings

This cell sets the execution environment, mounts Google Drive when available, and creates a reproducible output structure for an Array-ready benchmark pipeline.






In [ ]:
import os, json, math, random, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')

USE_GOOGLE_DRIVE = True
PROJECT_NAME = 'Array_Calibration_Aware_Pneumonia_Pipeline'
RANDOM_STATE = 42
SEED_LIST = [42, 123, 7, 2026, 314]

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from sklearn.metrics import (
        accuracy_score, balanced_accuracy_score, precision_score, recall_score,
        f1_score, roc_auc_score, average_precision_score, confusion_matrix,
        classification_report, brier_score_loss, log_loss
    )
    from sklearn.calibration import calibration_curve
    from sklearn.isotonic import IsotonicRegression
    from sklearn.linear_model import LogisticRegression
    from scipy.optimize import minimize_scalar
    from scipy.spatial.distance import cosine
except Exception as e:
    raise RuntimeError(f'Missing dependency: {e}')

os.environ.setdefault('PYTHONHASHSEED', str(RANDOM_STATE))
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
try:
    # Keras >= 2.7 seeds Python, NumPy and TensorFlow together in one call.
    keras.utils.set_random_seed(RANDOM_STATE)
except Exception:
    pass

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        BASE_DIR = Path('/content/drive/MyDrive/Outputs') / PROJECT_NAME
        DATASET_PATH = Path('/content/drive/MyDrive/Datasets/Pneumonia/pneumoniamnist.npz')
    except Exception as e:
        print('Google Drive mount failed; using local /content paths instead.')
        print('Reason:', e)
        BASE_DIR = Path('/content') / PROJECT_NAME
        DATASET_PATH = Path('/content/drive/MyDrive/Datasets/Pneumonia/pneumoniamnist.npz')
else:
    BASE_DIR = Path('/content') / PROJECT_NAME
    DATASET_PATH = Path('/content/drive/MyDrive/Datasets/Pneumonia/pneumoniamnist.npz')

FIG_DIR = BASE_DIR / 'figures'
TABLE_DIR = BASE_DIR / 'tables'
OUTPUT_DIR = BASE_DIR / 'outputs'
LOG_DIR = BASE_DIR / 'logs'
for d in [BASE_DIR, FIG_DIR, TABLE_DIR, OUTPUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = OUTPUT_DIR / 'outputs_summary.txt'
INTERPRET_PATH = OUTPUT_DIR / 'interpretation_summary.txt'

def log_message(msg):
    stamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{stamp}] {msg}'
    print(line)
    with open(SUMMARY_PATH, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

log_message(f'Base directory: {BASE_DIR}')
log_message(f'Dataset path: {DATASET_PATH}')
log_message(f'Dataset exists: {DATASET_PATH.exists()}')
log_message(f'TensorFlow version: {tf.__version__}')





## 2. Load PneumoniaMNIST from Drive

The notebook expects a `.npz` file containing train/validation/test images and labels. Images are normalized and reshaped for CNN input.





In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found at {DATASET_PATH}. Please verify the file exists in MyDrive/Datasets/Pneumonia/.'
    )

data = np.load(DATASET_PATH)
log_message(f'Available keys: {list(data.keys())}')

x_train = data['train_images']
y_train = data['train_labels'].reshape(-1).astype(int)
x_val = data['val_images']
y_val = data['val_labels'].reshape(-1).astype(int)
x_test = data['test_images']
y_test = data['test_labels'].reshape(-1).astype(int)

# Normalize and add channel dimension
x_train = x_train.astype('float32') / 255.0
x_val = x_val.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
if x_train.ndim == 3:
    x_train = x_train[..., None]
    x_val = x_val[..., None]
    x_test = x_test[..., None]

INPUT_SHAPE = x_train.shape[1:]
N_CLASSES = len(np.unique(y_train))

log_message(f'Train images: {x_train.shape}, labels: {y_train.shape}')
log_message(f'Validation images: {x_val.shape}, labels: {y_val.shape}')
log_message(f'Test images: {x_test.shape}, labels: {y_test.shape}')
log_message(f'Train label distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}')
log_message(f'Test label distribution: {dict(zip(*np.unique(y_test, return_counts=True)))}')
log_message(f'Input shape: {INPUT_SHAPE}; number of classes: {N_CLASSES}')





## 3. Visual inspection of samples

A small image grid is saved for traceability and manuscript illustration.





In [ ]:
label_names = {0: 'Normal', 1: 'Pneumonia'}
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for cls in [0, 1]:
    idxs = np.where(y_train == cls)[0][:8]
    for j, idx in enumerate(idxs):
        ax = axes[cls, j]
        ax.imshow(x_train[idx].squeeze(), cmap='gray')
        ax.set_title(label_names.get(cls, str(cls)), fontsize=9)
        ax.axis('off')
plt.tight_layout()
sample_path = FIG_DIR / 'sample_pneumonia_images.png'
plt.savefig(sample_path, dpi=300, bbox_inches='tight')
plt.show()
log_message(f'Saved sample image grid to {sample_path}')





## 4. Build a compact CNN with dropout

Dropout is intentionally retained because it will later be activated at inference time to estimate epistemic uncertainty through Monte Carlo dropout.





In [ ]:
def build_cnn(input_shape=INPUT_SHAPE, dropout_rate=0.35):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    logits = layers.Dense(1, name='logits')(x)
    outputs = layers.Activation('sigmoid', name='probability')(logits)
    model = keras.Model(inputs, outputs, name='Array_Pneumonia_CNN')
    return model



def build_ensemble(n=5, input_shape=INPUT_SHAPE, dropout_rate=0.35, seeds=None):
    """Create independently initialized CNN models for ensemble uncertainty."""
    seeds = SEED_LIST[:n] if seeds is None else seeds[:n]
    ensemble = []
    for seed in seeds:
        np.random.seed(seed)
        random.seed(seed)
        tf.random.set_seed(seed)
        member = build_cnn(input_shape=input_shape, dropout_rate=dropout_rate)
        member.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss='binary_crossentropy',
            metrics=['accuracy', keras.metrics.AUC(name='auc')]
        )
        ensemble.append(member)
    tf.random.set_seed(RANDOM_STATE)
    return ensemble

model = build_cnn()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)
model.summary()





## 5. Train with class weighting and early stopping

Because pneumonia datasets are usually imbalanced, class weights are used to reduce majority-class bias. This is important because balanced accuracy collapsed in the first run.





In [ ]:
classes, counts = np.unique(y_train, return_counts=True)
class_weight = {int(c): float(len(y_train) / (len(classes) * cnt)) for c, cnt in zip(classes, counts)}
log_message(f'Class weights: {class_weight}')

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=4, min_lr=1e-5)
]

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=40,
    batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

hist_df = pd.DataFrame(history.history)
hist_path = TABLE_DIR / 'training_history.csv'
hist_df.to_csv(hist_path, index=False)
log_message(f'Saved training history to {hist_path}')

for metric in ['loss', 'accuracy', 'auc']:
    plt.figure(figsize=(6,4))
    plt.plot(hist_df[metric], label=f'train_{metric}')
    if f'val_{metric}' in hist_df:
        plt.plot(hist_df[f'val_{metric}'], label=f'val_{metric}')
    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.legend()
    plt.tight_layout()
    out = FIG_DIR / f'training_curve_{metric}.png'
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.show()
    log_message(f'Saved {metric} curve to {out}')





## 6. Predictive evaluation before calibration

This cell evaluates raw CNN probabilities. These metrics establish the baseline evidence before calibration, threshold optimization, uncertainty analysis, and explainable evaluation are applied.





In [ ]:
def predict_prob(m, x):
    return m.predict(x, batch_size=256, verbose=0).reshape(-1)

p_val_raw = predict_prob(model, x_val)
p_test_raw = predict_prob(model, x_test)

# Raw threshold chosen on validation to maximize balanced accuracy.
threshold_grid = np.linspace(0.05, 0.95, 181)
val_baccs = [balanced_accuracy_score(y_val, (p_val_raw >= t).astype(int)) for t in threshold_grid]
RAW_OPT_THR = float(threshold_grid[int(np.argmax(val_baccs))])
log_message(f'Raw validation-optimized threshold: {RAW_OPT_THR:.3f}')

raw_pred = (p_test_raw >= RAW_OPT_THR).astype(int)
raw_metrics = {
    'threshold': RAW_OPT_THR,
    'accuracy': accuracy_score(y_test, raw_pred),
    'balanced_accuracy': balanced_accuracy_score(y_test, raw_pred),
    'precision': precision_score(y_test, raw_pred, zero_division=0),
    'recall': recall_score(y_test, raw_pred, zero_division=0),
    'f1': f1_score(y_test, raw_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, p_test_raw),
    'average_precision': average_precision_score(y_test, p_test_raw),
    'brier_score': brier_score_loss(y_test, p_test_raw)
}
raw_df = pd.DataFrame([raw_metrics])
raw_path = TABLE_DIR / 'raw_predictive_metrics.csv'
raw_df.to_csv(raw_path, index=False)
print(raw_df.round(4))
log_message(f'Saved raw predictive metrics to {raw_path}')

cm = confusion_matrix(y_test, raw_pred)
plt.figure(figsize=(4,4))
plt.imshow(cm)
plt.title('Raw Confusion Matrix')
plt.xticks([0,1], ['Normal','Pneumonia'])
plt.yticks([0,1], ['Normal','Pneumonia'])
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i,j]), ha='center', va='center')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
cm_path = FIG_DIR / 'raw_confusion_matrix.png'
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.show()
log_message(f'Saved raw confusion matrix to {cm_path}')





## 7. Calibration layer: ranking-preserving primary policy and method comparison

The pipeline avoids using raw neural probabilities when calibration is poor. Temperature scaling transforms logits before probability conversion:

\[
\hat{p}_{cal}(y=1\mid x) = \sigma\left(\frac{z(x)}{T}\right), \qquad T>0
\]

where \(z(x)\) is the logit associated with the raw model probability and \(T\) is optimized on the validation set by minimizing negative log-likelihood.

For the Array version, temperature scaling is used as the primary calibrated model because it is monotonic and therefore preserves ranking-sensitive metrics such as ROC-AUC. Platt scaling and isotonic regression are still evaluated and reported, but they are treated as comparative calibration baselines. This avoids the misleading situation in which a non-parametric calibrator improves a validation calibration score while degrading test ranking performance.



In [ ]:
# =========================================================
# Calibration comparison with ranking-preserving primary policy
# =========================================================

CALIBRATION_SELECTION_POLICY = 'ranking_preserving_temperature_primary'
MIN_VALIDATION_ECE_IMPROVEMENT = 0.005
MAX_VALIDATION_AUC_DROP = 0.005


def prob_to_logit(p, eps=1e-7):
    p = np.clip(np.asarray(p, dtype=float), eps, 1 - eps)
    return np.log(p / (1 - p))


def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1 / (1 + np.exp(-z))


def nll_temperature(T, logits, y):
    p = sigmoid(logits / T)
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


def safe_roc_auc(y_true, prob):
    try:
        return float(roc_auc_score(y_true, prob))
    except Exception:
        return np.nan


def calibration_bins(y_true, prob, n_bins=10):
    """Robust uniform-bin calibration table and ECE.

    Empty bins are kept in the exported table for transparency, but they do not
    contribute to ECE. This prevents NaN propagation in final manuscript summaries.
    """
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    prob = np.asarray(prob, dtype=float).reshape(-1)
    prob = np.clip(prob, 0.0, 1.0)
    if len(y_true) != len(prob):
        raise ValueError(f'y_true and prob must have the same length; got {len(y_true)} and {len(prob)}')
    if len(y_true) == 0:
        raise ValueError('Cannot compute calibration metrics on an empty array.')

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (prob >= lo) & ((prob < hi) if i < n_bins - 1 else (prob <= hi))
        n = int(mask.sum())
        if n == 0:
            rows.append({'bin': i + 1, 'lo': lo, 'hi': hi, 'count': 0,
                         'confidence': np.nan, 'empirical_rate': np.nan,
                         'gap': np.nan, 'signed_gap': np.nan,
                         'ece_contribution': 0.0})
            continue
        conf = float(prob[mask].mean())
        emp = float(y_true[mask].mean())
        signed_gap = conf - emp
        gap = abs(signed_gap)
        contribution = (n / len(y_true)) * gap
        ece += contribution
        rows.append({'bin': i + 1, 'lo': lo, 'hi': hi, 'count': n,
                     'confidence': conf, 'empirical_rate': emp,
                     'gap': gap, 'signed_gap': signed_gap,
                     'ece_contribution': contribution})
    return float(ece), pd.DataFrame(rows)


def calibration_metrics(y_true, prob, n_bins=10):
    ece, bins_df = calibration_bins(y_true, prob, n_bins=n_bins)
    valid = bins_df.dropna(subset=['gap'])
    mce = float(valid['gap'].max()) if len(valid) else np.nan
    overconfidence_ratio = float((valid['signed_gap'] > 0).mean()) if len(valid) else np.nan
    sharpness = float(np.mean(np.abs(np.asarray(prob) - 0.5)) * 2.0)
    return {
        'ece': ece,
        'mce': mce,
        'overconfidence_ratio': overconfidence_ratio,
        'sharpness': sharpness,
        'brier_score': brier_score_loss(y_true, prob),
        'nll': log_loss(y_true, np.clip(prob, 1e-7, 1 - 1e-7)),
        'roc_auc': safe_roc_auc(y_true, prob),
    }, bins_df


val_logits = prob_to_logit(p_val_raw)
test_logits = prob_to_logit(p_test_raw)

# 1) Temperature scaling: primary ranking-preserving calibration method
res = minimize_scalar(lambda T: nll_temperature(T, val_logits, y_val), bounds=(0.2, 10.0), method='bounded')
TEMP = float(res.x)
p_val_temp = sigmoid(val_logits / TEMP)
p_test_temp = sigmoid(test_logits / TEMP)

# 2) Platt scaling on logits: comparative monotonic calibration baseline
platt_model = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE)
platt_model.fit(val_logits.reshape(-1, 1), y_val)
p_val_platt = platt_model.predict_proba(val_logits.reshape(-1, 1))[:, 1]
p_test_platt = platt_model.predict_proba(test_logits.reshape(-1, 1))[:, 1]

# 3) Isotonic regression on raw probabilities: flexible but not used as primary by default
isotonic_model = IsotonicRegression(out_of_bounds='clip')
isotonic_model.fit(p_val_raw, y_val)
p_val_isotonic = isotonic_model.predict(p_val_raw)
p_test_isotonic = isotonic_model.predict(p_test_raw)

calibration_candidates = {
    'raw_uncalibrated': (p_val_raw, p_test_raw, True),
    'temperature_scaling': (p_val_temp, p_test_temp, True),
    'platt_scaling': (p_val_platt, p_test_platt, True),
    'isotonic_regression': (p_val_isotonic, p_test_isotonic, False),
}

raw_val_metrics, _ = calibration_metrics(y_val, p_val_raw, n_bins=10)
raw_val_auc = raw_val_metrics['roc_auc']

calibration_rows = []
calibration_val_probs = {}
calibration_test_probs = {}
calibration_bins_dict = {}
for method, (p_val_method, p_test_method, rank_preserving) in calibration_candidates.items():
    val_metrics, _ = calibration_metrics(y_val, p_val_method, n_bins=10)
    test_metrics, test_bins_method = calibration_metrics(y_test, p_test_method, n_bins=10)
    validation_auc_drop = raw_val_auc - val_metrics['roc_auc'] if not np.isnan(raw_val_auc) and not np.isnan(val_metrics['roc_auc']) else np.nan
    validation_ece_improvement = raw_val_metrics['ece'] - val_metrics['ece']
    calibration_rows.append({
        'method': method,
        'selection_policy': CALIBRATION_SELECTION_POLICY,
        'rank_preserving': rank_preserving,
        'validation_ece': val_metrics['ece'],
        'validation_ece_improvement_vs_raw': validation_ece_improvement,
        'validation_roc_auc': val_metrics['roc_auc'],
        'validation_auc_drop_vs_raw': validation_auc_drop,
        'test_ece': test_metrics['ece'],
        'test_mce': test_metrics['mce'],
        'test_overconfidence_ratio': test_metrics['overconfidence_ratio'],
        'test_sharpness': test_metrics['sharpness'],
        'test_brier_score': test_metrics['brier_score'],
        'test_nll': test_metrics['nll'],
        'test_roc_auc': test_metrics['roc_auc']
    })
    calibration_val_probs[method] = p_val_method
    calibration_test_probs[method] = p_test_method
    calibration_bins_dict[method] = test_bins_method.assign(method=method)

calibration_comparison_df = pd.DataFrame(calibration_rows).sort_values(['validation_ece', 'validation_auc_drop_vs_raw'])
VALIDATION_SELECTED_CALIBRATION_METHOD = str(
    calibration_comparison_df[calibration_comparison_df['method'] != 'raw_uncalibrated'].iloc[0]['method']
)

# Guarded final choice for manuscript reporting.
# Temperature scaling is used as the primary calibrated model because it is monotonic and ranking-preserving.
# Other calibrators remain in the comparison table for transparency and reproducibility.
FINAL_CALIBRATION_METHOD = 'temperature_scaling'
CALIBRATION_METHOD = FINAL_CALIBRATION_METHOD
p_val_cal = calibration_val_probs[FINAL_CALIBRATION_METHOD]
p_test_cal = calibration_test_probs[FINAL_CALIBRATION_METHOD]

cal_comparison_path = TABLE_DIR / 'calibration_method_comparison.csv'
calibration_comparison_df.to_csv(cal_comparison_path, index=False)
cal_guarded_path = TABLE_DIR / 'calibration_method_comparison_guarded.csv'
calibration_comparison_df.to_csv(cal_guarded_path, index=False)

log_message(f'Saved calibration method comparison to {cal_comparison_path}')
log_message(f'Validation-selected calibration method by ECE: {VALIDATION_SELECTED_CALIBRATION_METHOD}')
log_message(f'Final manuscript calibration method: {FINAL_CALIBRATION_METHOD} ({CALIBRATION_SELECTION_POLICY})')
log_message(f'Temperature-scaling parameter: {TEMP:.4f}; validation NLL: {res.fun:.4f}')

cal_baccs = [balanced_accuracy_score(y_val, (p_val_cal >= t).astype(int)) for t in threshold_grid]
CAL_OPT_THR = float(threshold_grid[int(np.argmax(cal_baccs))])
log_message(f'Calibrated validation-optimized threshold: {CAL_OPT_THR:.3f}')

cal_pred = (p_test_cal >= CAL_OPT_THR).astype(int)
cal_metrics = {
    'calibration_method': FINAL_CALIBRATION_METHOD,
    'validation_selected_method_by_ece': VALIDATION_SELECTED_CALIBRATION_METHOD,
    'selection_policy': CALIBRATION_SELECTION_POLICY,
    'temperature_parameter': TEMP,
    'threshold': CAL_OPT_THR,
    'accuracy': accuracy_score(y_test, cal_pred),
    'balanced_accuracy': balanced_accuracy_score(y_test, cal_pred),
    'precision': precision_score(y_test, cal_pred, zero_division=0),
    'recall': recall_score(y_test, cal_pred, zero_division=0),
    'f1': f1_score(y_test, cal_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, p_test_cal),
    'average_precision': average_precision_score(y_test, p_test_cal),
    'brier_score': brier_score_loss(y_test, p_test_cal),
    'nll': log_loss(y_test, np.clip(p_test_cal, 1e-7, 1 - 1e-7))
}
cal_df = pd.DataFrame([cal_metrics])
cal_path = TABLE_DIR / 'calibrated_predictive_metrics.csv'
cal_df.to_csv(cal_path, index=False)
print(calibration_comparison_df.round(4))
print(cal_df.round(4))
log_message(f'Saved calibrated predictive metrics to {cal_path}')



## 8. Calibration diagnostics: reliability, ECE, MCE, and heatmap

Expected Calibration Error (ECE) measures how far confidence is from empirical correctness. Lower is safer for decision support.





In [ ]:
# =========================================================
# Extended calibration diagnostics: ECE, MCE, sharpness, and calibration heatmap
# =========================================================

raw_metrics_cal, raw_bins = calibration_metrics(y_test, p_test_raw, n_bins=10)
cal_metrics_cal, cal_bins = calibration_metrics(y_test, p_test_cal, n_bins=10)
raw_bins.to_csv(TABLE_DIR / 'raw_calibration_bins.csv', index=False)
cal_bins.to_csv(TABLE_DIR / 'calibrated_calibration_bins.csv', index=False)

all_calibration_bins = pd.concat([df for df in calibration_bins_dict.values()], ignore_index=True)
all_calibration_bins.to_csv(TABLE_DIR / 'calibration_bins_extended.csv', index=False)


# Explicit manuscript-safe references used by the final summary cell.
RAW_ECE_REFERENCE = float(raw_metrics_cal['ece'])
FINAL_CAL_ECE = float(cal_metrics_cal['ece'])
RAW_MCE_REFERENCE = float(raw_metrics_cal['mce'])
FINAL_CAL_MCE = float(cal_metrics_cal['mce'])

if not np.isfinite(RAW_ECE_REFERENCE) or not np.isfinite(FINAL_CAL_ECE):
    raise ValueError('ECE computation produced a non-finite value. Check probability arrays and calibration bins.')

extended_calibration_summary = pd.DataFrame([
    {'method': 'raw_uncalibrated', **raw_metrics_cal},
    {'method': CALIBRATION_METHOD, **cal_metrics_cal}
])
extended_calibration_path = TABLE_DIR / 'extended_calibration_diagnostics.csv'
extended_calibration_summary.to_csv(extended_calibration_path, index=False)

log_message(f'Raw ECE: {raw_metrics_cal["ece"]:.4f}')
log_message(f'Calibrated ECE ({CALIBRATION_METHOD}): {cal_metrics_cal["ece"]:.4f}')
log_message(f'Saved extended calibration diagnostics to {extended_calibration_path}')

plt.figure(figsize=(5.5, 5))
for prob, name in [(p_test_raw, 'Raw'), (p_test_cal, CALIBRATION_METHOD.replace('_', ' ').title())]:
    frac_pos, mean_pred = calibration_curve(y_test, prob, n_bins=10, strategy='uniform')
    plt.plot(mean_pred, frac_pos, marker='o', label=name)
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect calibration')
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed pneumonia frequency')
plt.title('Reliability Curve')
plt.legend()
plt.tight_layout()
rel_path = FIG_DIR / 'reliability_curve_raw_vs_calibrated.png'
plt.savefig(rel_path, dpi=300, bbox_inches='tight')
plt.show()
log_message(f'Saved reliability curve to {rel_path}')

heat_df = cal_bins.copy()
heat_df['bin_label'] = heat_df['bin'].astype(str)
heat_values = heat_df[['confidence', 'empirical_rate', 'gap']].to_numpy().T
plt.figure(figsize=(8, 3.4))
plt.imshow(heat_values, aspect='auto')
plt.yticks([0, 1, 2], ['Confidence', 'Empirical accuracy', 'Absolute gap'])
plt.xticks(range(len(heat_df)), heat_df['bin_label'])
plt.xlabel('Calibration bin')
plt.title('Calibration Heatmap')
plt.colorbar(label='Value')
plt.tight_layout()
heatmap_path = FIG_DIR / 'calibration_heatmap.png'
plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
plt.show()
log_message(f'Saved calibration heatmap to {heatmap_path}')






### Submission check: calibration diagnostics are now finite

The calibration diagnostics cell explicitly exports raw and calibrated ECE values and raises an error if either value becomes non-finite. This prevents a manuscript summary with `Raw ECE: nan` and makes the notebook safer for reproducible execution and repository release.


## 9. MC-dropout uncertainty estimation

Dropout is activated during inference to obtain a distribution of predictions. The mean gives the prediction, while entropy and standard deviation describe uncertainty.





In [ ]:
# =========================================================
# MC-dropout uncertainty estimation and uncertainty-stratified diagnostics
# =========================================================

def mc_dropout_predict(m, x, n_passes=100, batch_size=256):
    """Stochastic forward passes with dropout active at inference time.

    Dropout masks are resampled on every forward call, so batching the input
    only changes how cases are grouped within a pass; it does not change the
    sampling distribution. Honouring ``batch_size`` keeps memory bounded for
    larger inputs.
    """
    n = len(x)
    preds = np.empty((n_passes, n), dtype=np.float32)
    for i in range(n_passes):
        if (i + 1) % 10 == 0:
            print(f'MC pass {i + 1}/{n_passes}')
        chunks = []
        for start in range(0, n, batch_size):
            xb = x[start:start + batch_size]
            chunks.append(m(xb, training=True).numpy().reshape(-1))
        preds[i] = np.concatenate(chunks)
    return preds

def entropy_binary(p, eps=1e-7):
    p = np.clip(p, eps, 1 - eps)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p)) / np.log(2)

MC_DROPOUT_PASSES = 100
mc_raw = mc_dropout_predict(model, x_test, n_passes=MC_DROPOUT_PASSES)

if CALIBRATION_METHOD == 'temperature_scaling':
    mc_cal = sigmoid(prob_to_logit(mc_raw) / TEMP)
elif CALIBRATION_METHOD == 'platt_scaling':
    mc_cal = platt_model.predict_proba(prob_to_logit(mc_raw).reshape(-1, 1))[:, 1].reshape(mc_raw.shape)
elif CALIBRATION_METHOD == 'isotonic_regression':
    mc_cal = isotonic_model.predict(mc_raw.reshape(-1)).reshape(mc_raw.shape)
else:
    raise ValueError(f'Unknown calibration method: {CALIBRATION_METHOD}')

mc_mean = mc_cal.mean(axis=0)
mc_std = mc_cal.std(axis=0)
mc_entropy = entropy_binary(mc_mean)

unc_df = pd.DataFrame({
    'y_true': y_test,
    'p_raw': p_test_raw,
    'p_calibrated': p_test_cal,
    'p_mc_calibrated_mean': mc_mean,
    'mc_std': mc_std,
    'predictive_entropy': mc_entropy,
    'raw_prediction': raw_pred,
    'calibrated_prediction': cal_pred
})
unc_path = TABLE_DIR / 'mc_dropout_uncertainty_calibrated.csv'
unc_df.to_csv(unc_path, index=False)
log_message(f'Saved calibrated MC-dropout uncertainty table to {unc_path}')
log_message(f'Mean predictive entropy: {mc_entropy.mean():.4f}')
log_message(f'Mean MC std: {mc_std.mean():.4f}')

unc_df['entropy_quartile'] = pd.qcut(unc_df['predictive_entropy'].rank(method='first'), q=4,
                                     labels=['Q1_lowest_uncertainty', 'Q2', 'Q3', 'Q4_highest_uncertainty'])
quartile_rows = []
for q, sub in unc_df.groupby('entropy_quartile'):
    prob = np.clip(sub['p_mc_calibrated_mean'].to_numpy(), 1e-7, 1 - 1e-7)
    yy = sub['y_true'].to_numpy().astype(int)
    pred = (prob >= CAL_OPT_THR).astype(int)
    quartile_rows.append({
        'uncertainty_quartile': str(q),
        'n_cases': len(sub),
        'mean_entropy': float(sub['predictive_entropy'].mean()),
        'mean_mc_std': float(sub['mc_std'].mean()),
        'nll': log_loss(yy, prob),
        'brier_score': brier_score_loss(yy, prob),
        'balanced_accuracy': balanced_accuracy_score(yy, pred),
        'false_negatives': int(((yy == 1) & (pred == 0)).sum()),
        'false_positives': int(((yy == 0) & (pred == 1)).sum())
    })
quartile_df = pd.DataFrame(quartile_rows)
quartile_path = TABLE_DIR / 'uncertainty_quartile_stratification.csv'
quartile_df.to_csv(quartile_path, index=False)
print(quartile_df.round(4))
log_message(f'Saved uncertainty quartile stratification to {quartile_path}')

plt.figure(figsize=(6, 4))
plt.hist(mc_entropy, bins=25)
plt.xlabel('Predictive entropy')
plt.ylabel('Number of cases')
plt.title('Uncertainty Distribution')
plt.tight_layout()
ent_path = FIG_DIR / 'predictive_entropy_distribution.png'
plt.savefig(ent_path, dpi=300, bbox_inches='tight')
plt.show()
log_message(f'Saved uncertainty distribution to {ent_path}')






## Ensemble uncertainty reference

This optional cell trains a small five-member CNN ensemble to provide an epistemic uncertainty reference against MC dropout. It is disabled by default because it is computationally heavier than the main workflow. Enable it when preparing final manuscript tables or extended reviewer material.






In [ ]:
# =========================================================
# Optional deep ensemble uncertainty reference
# =========================================================

RUN_DEEP_ENSEMBLE = False
ENSEMBLE_MEMBERS = 5

if RUN_DEEP_ENSEMBLE:
    ensemble_models = build_ensemble(n=ENSEMBLE_MEMBERS)
    ensemble_probs = []
    for member_id, member in enumerate(ensemble_models, start=1):
        log_message(f'Training ensemble member {member_id}/{ENSEMBLE_MEMBERS}.')
        callbacks = [
            keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=6, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=3, min_lr=1e-5)
        ]
        member.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=30, batch_size=64,
                   class_weight=class_weight, callbacks=callbacks, verbose=1)
        ensemble_probs.append(predict_prob(member, x_test))

    ensemble_probs = np.vstack(ensemble_probs)
    ensemble_mean_raw = ensemble_probs.mean(axis=0)
    if CALIBRATION_METHOD == 'temperature_scaling':
        ensemble_mean = sigmoid(prob_to_logit(ensemble_mean_raw) / TEMP)
    elif CALIBRATION_METHOD == 'platt_scaling':
        ensemble_mean = platt_model.predict_proba(prob_to_logit(ensemble_mean_raw).reshape(-1, 1))[:, 1]
    elif CALIBRATION_METHOD == 'isotonic_regression':
        ensemble_mean = isotonic_model.predict(ensemble_mean_raw)
    else:
        ensemble_mean = ensemble_mean_raw
    ensemble_std = ensemble_probs.std(axis=0)
    ensemble_entropy = entropy_binary(ensemble_mean)
    ensemble_ece, _ = calibration_bins(y_test, ensemble_mean, n_bins=10)
    uncertainty_method_df = pd.DataFrame([
        {'method': 'mc_dropout', 'ece': cal_metrics_cal['ece'], 'nll': log_loss(y_test, np.clip(mc_mean, 1e-7, 1 - 1e-7)), 'brier_score': brier_score_loss(y_test, mc_mean), 'mean_entropy': float(mc_entropy.mean()), 'mean_std': float(mc_std.mean())},
        {'method': 'deep_ensemble', 'ece': ensemble_ece, 'nll': log_loss(y_test, np.clip(ensemble_mean, 1e-7, 1 - 1e-7)), 'brier_score': brier_score_loss(y_test, ensemble_mean), 'mean_entropy': float(ensemble_entropy.mean()), 'mean_std': float(ensemble_std.mean())}
    ])
else:
    uncertainty_method_df = pd.DataFrame([
        {'method': 'mc_dropout', 'ece': cal_metrics_cal['ece'], 'nll': log_loss(y_test, np.clip(mc_mean, 1e-7, 1 - 1e-7)), 'brier_score': brier_score_loss(y_test, mc_mean), 'mean_entropy': float(mc_entropy.mean()), 'mean_std': float(mc_std.mean())},
        {'method': 'deep_ensemble', 'ece': np.nan, 'nll': np.nan, 'brier_score': np.nan, 'mean_entropy': np.nan, 'mean_std': np.nan, 'note': 'Set RUN_DEEP_ENSEMBLE=True to train the ensemble reference.'}
    ])
uncertainty_method_path = TABLE_DIR / 'uncertainty_method_comparison.csv'
uncertainty_method_df.to_csv(uncertainty_method_path, index=False)
print(uncertainty_method_df)
log_message(f'Saved uncertainty method comparison to {uncertainty_method_path}')






## 10. Multi-Criteria Decision Score (MCDS) for structured evaluation

The MCDS layer is not a new medical theory and not a replacement for workflow diagnosis. It is a reproducible structured evaluation criterion used to compare candidate review actions under calibrated probability, uncertainty, calibration residuals, threshold sensitivity, and workflow cost.

For each case \(x_i\) and candidate action \(a\), the selected structured action minimizes:

\[
\mathcal{S}_{\mathrm{MCDS}}(a \mid x_i) =
 w_r R(a,x_i) + w_u U_a(x_i) + w_c C(x_i)
 + w_s S_a + w_b B(a) + w_a A(a)
 + w_p D_a(x_i) + \Phi_a(x_i) - w_i I_a(x_i).
\]

A hard feasibility gate is applied before score minimization. Low-risk monitoring is treated as workflow-infeasible when pneumonia probability or predictive uncertainty exceeds conservative limits. This keeps the notebook aligned with an Array-style reproducible evaluation framework: transparent criteria first, threshold sensitivity second, and safety-oriented audit tables for reporting.

The main terms are:

- \(D_a(x_i)\): distance between the case profile and the expected operating region of action \(a\).
- \(\Phi_a(x_i)\): soft safety barrier penalizing low-intensity actions under elevated risk or uncertainty.
- \(I_a(x_i)\): action-specific information or confidence reward.
- \(\mathcal{G}(x_i)\): hard feasibility gate that prevents low-risk monitoring above predefined probability or uncertainty limits.





In [ ]:

ACTIONS = [
    'Low-risk monitoring / outpatient follow-up',
    'Treat as likely pneumonia with clinician confirmation',
    'Request additional evidence or repeat imaging',
    'Clinician review recommended',
    'Escalate review: high-risk but uncertain'
]

# Action-level profiles are normalized design variables. They express where each
# review action is expected to operate in probability-uncertainty space.
action_props = pd.DataFrame({
    'action': ACTIONS,
    'resource_burden': [0.08, 0.32, 0.50, 0.42, 0.82],
    'action_complexity': [0.08, 0.36, 0.48, 0.45, 0.88],
    'information_gain': [0.04, 0.38, 0.82, 0.72, 0.90],
    'ideal_probability': [0.10, 0.86, 0.50, 0.62, 0.90],
    'ideal_uncertainty': [0.10, 0.16, 0.62, 0.46, 0.66],
    'probability_width': [0.26, 0.28, 0.36, 0.34, 0.24],
    'uncertainty_width': [0.30, 0.26, 0.32, 0.30, 0.34]
})

action_profiles_path = TABLE_DIR / 'mcds_action_profiles.csv'
action_props.to_csv(action_profiles_path, index=False)
log_message(f'Saved MCDS action profiles to {action_profiles_path}')

def calibration_residual(prob, bins_df):
    # Assign each case the absolute confidence-observation gap of its calibration bin.
    gaps = []
    for p in prob:
        row = bins_df[(p >= bins_df['lo']) & (p <= bins_df['hi'])].head(1)
        if len(row) == 0 or pd.isna(row['gap'].iloc[0]):
            gaps.append(0.0)
        else:
            gaps.append(float(row['gap'].iloc[0]))
    return np.array(gaps)

cal_resid = calibration_residual(mc_mean, cal_bins)

# Normalized uncertainty components. Binary entropy has maximum ln(2).
entropy_norm = np.clip(mc_entropy, 0, 1)  # mc_entropy is already in bits ([0, 1])
std_norm = np.clip(mc_std / max(float(np.nanpercentile(mc_std, 95)), 1e-6), 0, 1)
combined_uncertainty = np.clip(0.72 * entropy_norm + 0.28 * std_norm, 0, 1)
confidence_strength = np.clip(2 * np.abs(mc_mean - 0.5), 0, 1)
diagnostic_ambiguity = np.clip(4 * mc_mean * (1 - mc_mean), 0, 1)

# Baseline scenario. These weights are intentionally rebalanced to preserve
# safety while allowing differentiated action selection.
SCENARIO = {
    'name': 'baseline_workflow_pressure',
    'seasonal_pressure': 0.40,
    'false_negative_cost': 1.10,
    'false_positive_cost': 0.42,
    'weights': {
        'risk_cost': 1.00,
        'uncertainty': 0.45,
        'calibration': 0.42,
        'seasonal_pressure': 0.22,
        'resource_burden': 0.28,
        'action_complexity': 0.20,
        'profile_distance': 0.82,
        'information_gain': 0.50,
        'confidence_reward': 0.72,
        'safety_barrier': 1.75
    }
}


# Decision-support safety-feasibility gate.
# Low-risk monitoring is not allowed when either pneumonia probability or
# uncertainty exceeds these conservative limits. This is not a learned label rule;
# it is a workflow-motivated action-feasibility constraint applied before the
# MCDS argmin selection.
SAFETY_FEASIBILITY = {
    'low_monitoring_max_probability': 0.34,
    'low_monitoring_max_uncertainty': 0.45,
    'infeasible_penalty': 1e6
}

def positive_part(x):
    return np.maximum(x, 0.0)

def mcds_action_scores(prob, entropy, std, cal_gap, scenario=SCENARIO):
    w = scenario['weights']
    s_pressure = scenario['seasonal_pressure']
    fn_cost = scenario['false_negative_cost']
    fp_cost = scenario['false_positive_cost']

    # Normalize uncertainty inside the function to support scenario re-use.
    e_norm = np.clip(entropy, 0, 1)  # entropy_binary already returns bits in [0, 1]
    s_norm = np.clip(std / max(float(np.nanpercentile(std, 95)), 1e-6), 0, 1)
    u = np.clip(0.72 * e_norm + 0.28 * s_norm, 0, 1)
    confidence = np.clip(2 * np.abs(prob - 0.5), 0, 1)
    ambiguity = np.clip(4 * prob * (1 - prob), 0, 1)

    rows = []
    for i, p in enumerate(prob):
        for _, arow in action_props.iterrows():
            action = arow['action']
            burden = float(arow['resource_burden'])
            complexity = float(arow['action_complexity'])
            info_gain = float(arow['information_gain'])
            p0 = float(arow['ideal_probability'])
            u0 = float(arow['ideal_uncertainty'])
            pw = float(arow['probability_width'])
            uw = float(arow['uncertainty_width'])

            # Profile distance separates the natural operating regions of the actions.
            profile_distance = ((p - p0) / pw) ** 2 + ((u[i] - u0) / uw) ** 2

            # Workflow risk terms are action dependent.
            if 'Low-risk monitoring' in action:
                risk_cost = fn_cost * (p ** 1.30) + 0.25 * u[i]
                uncertainty_term = 0.90 * u[i]
                info_reward = 0.12 * (1 - p) * (1 - u[i])
                confidence_reward = confidence[i] * (1 - p)
                safety_barrier = positive_part(p - 0.34) ** 2 + 0.65 * positive_part(u[i] - 0.45) ** 2
                seasonal_term = s_pressure * p

                feasibility_violation = int(
                    (p > SAFETY_FEASIBILITY['low_monitoring_max_probability'])
                    or (u[i] > SAFETY_FEASIBILITY['low_monitoring_max_uncertainty'])
                )
                feasibility_penalty = (
                    SAFETY_FEASIBILITY['infeasible_penalty']
                    if feasibility_violation else 0.0
                )

            elif 'Treat as likely pneumonia' in action:
                risk_cost = fp_cost * ((1 - p) ** 1.20) + 0.30 * u[i]
                uncertainty_term = 0.55 * u[i]
                info_reward = 0.42 * p * (1 - u[i])
                confidence_reward = confidence[i] * p * (1 - u[i])
                safety_barrier = 0.85 * positive_part(0.55 - p) ** 2 + 0.85 * positive_part(u[i] - 0.55) ** 2
                seasonal_term = -0.25 * s_pressure * p

            elif 'Request additional evidence' in action:
                risk_cost = 0.22 * ambiguity[i] + 0.08 * abs(p - 0.5)
                uncertainty_term = 0.18 * abs(u[i] - 0.62)
                info_reward = 0.50 * (0.55 * ambiguity[i] + 0.45 * u[i])
                confidence_reward = 0.18 * (1 - confidence[i])
                safety_barrier = 0.05 * positive_part(p - 0.92) ** 2
                seasonal_term = 0.04 * s_pressure

            elif 'Clinician review' in action:
                risk_cost = 0.18 * ambiguity[i] + 0.18 * u[i] + 0.16 * cal_gap[i]
                uncertainty_term = 0.24 * u[i]
                info_reward = 0.42 * (0.45 * u[i] + 0.35 * ambiguity[i] + 0.20 * cal_gap[i])
                confidence_reward = 0.12 * confidence[i]
                safety_barrier = 0.04 * positive_part(p - 0.96) ** 2
                seasonal_term = 0.02 * s_pressure

            elif 'Escalate' in action:
                risk_cost = 0.12 * (1 - p) + 0.10 * ambiguity[i]
                uncertainty_term = 0.20 * u[i]
                info_reward = 0.58 * p * (0.55 * u[i] + 0.45 * s_pressure)
                confidence_reward = 0.22 * confidence[i] * p
                safety_barrier = 0.75 * positive_part(0.62 - p) ** 2
                seasonal_term = -0.35 * s_pressure * p
            else:
                risk_cost = ambiguity[i]
                uncertainty_term = u[i]
                info_reward = 0.0
                confidence_reward = 0.0
                safety_barrier = 0.0
                seasonal_term = 0.0

            if 'Low-risk monitoring' not in action:
                feasibility_violation = 0
                feasibility_penalty = 0.0

            score = (
                w['risk_cost'] * risk_cost
                + w['uncertainty'] * uncertainty_term
                + w['calibration'] * cal_gap[i]
                + w['seasonal_pressure'] * seasonal_term
                + w['resource_burden'] * burden
                + w['action_complexity'] * complexity
                + w['profile_distance'] * profile_distance
                + w['safety_barrier'] * safety_barrier
                + feasibility_penalty
                - w['information_gain'] * info_gain * info_reward
                - w['confidence_reward'] * confidence_reward
            )

            rows.append({
                'case_id': i,
                'y_true': int(y_test[i]),
                'pneumonia_probability_calibrated_mc': float(p),
                'predictive_entropy': float(entropy[i]),
                'predictive_uncertainty_normalized': float(u[i]),
                'confidence_strength': float(confidence[i]),
                'diagnostic_ambiguity': float(ambiguity[i]),
                'mc_std': float(std[i]),
                'calibration_gap': float(cal_gap[i]),
                'scenario': scenario['name'],
                'action': action,
                'risk_cost_component': float(risk_cost),
                'uncertainty_component': float(uncertainty_term),
                'profile_distance_component': float(profile_distance),
                'safety_barrier_component': float(safety_barrier),
                'safety_feasibility_violation': int(feasibility_violation),
                'safety_feasibility_penalty': float(feasibility_penalty),
                'seasonal_pressure_component': float(seasonal_term),
                'seasonal_pressure': float(s_pressure),
                'resource_burden': burden,
                'action_complexity': complexity,
                'information_gain': info_gain,
                'information_reward_component': float(info_reward),
                'confidence_reward_component': float(confidence_reward),
                'mcds_score': float(score)
            })
    return pd.DataFrame(rows)

score_df = mcds_action_scores(mc_mean, mc_entropy, mc_std, cal_resid, SCENARIO)
best_actions = score_df.loc[score_df.groupby('case_id')['mcds_score'].idxmin()].reset_index(drop=True)
score_path = TABLE_DIR / 'mcds_action_scores_rebalanced.csv'
best_path = TABLE_DIR / 'mcds_selected_actions_rebalanced.csv'
score_df.to_csv(score_path, index=False)
best_actions.to_csv(best_path, index=False)
log_message(f'Saved rebalanced MCDS action scores to {score_path}')
log_message(f'Saved rebalanced selected MCDS actions to {best_path}')
best_actions.head()







## 11. Structured action distribution

The output is analyzed both as a calibrated binary classifier and as a structured decision-support routing table. The action distribution is used for reproducible evaluation, not for autonomous diagnosis.





In [ ]:

action_dist = best_actions['action'].value_counts().rename_axis('action').reset_index(name='count')
action_dist['percentage'] = 100 * action_dist['count'] / len(best_actions)
action_dist['decision_entropy'] = -(action_dist['percentage']/100) * np.log((action_dist['percentage']/100).clip(1e-12, 1))
policy_entropy = float(action_dist['decision_entropy'].sum())
policy_effective_actions = float(np.exp(policy_entropy))
action_path = TABLE_DIR / 'mcds_action_distribution_rebalanced.csv'
action_dist.to_csv(action_path, index=False)
print(action_dist[['action','count','percentage']])
print(f'Policy entropy: {policy_entropy:.4f}; effective number of actions: {policy_effective_actions:.2f}')
log_message(f'Saved rebalanced MCDS action distribution to {action_path}')
log_message(f'Policy entropy: {policy_entropy:.4f}; effective number of actions: {policy_effective_actions:.2f}')

plt.figure(figsize=(9,4.5))
plt.bar(action_dist['action'], action_dist['count'])
plt.xticks(rotation=30, ha='right')
plt.ylabel('Number of cases')
plt.title('Rebalanced MCDS Workflow Action Distribution')
plt.tight_layout()
action_fig = FIG_DIR / 'mcds_workflow_action_distribution_rebalanced.png'
plt.savefig(action_fig, dpi=300, bbox_inches='tight')
plt.show()
log_message(f'Saved rebalanced MCDS action distribution figure to {action_fig}')

# Decision landscape: probability vs uncertainty colored by selected action.
plot_df = best_actions.copy()
plt.figure(figsize=(7.5,5.5))
for action in ACTIONS:
    sub = plot_df[plot_df['action'] == action]
    if len(sub) > 0:
        plt.scatter(sub['pneumonia_probability_calibrated_mc'], sub['predictive_uncertainty_normalized'], s=24, alpha=0.75, label=action)
plt.xlabel('Calibrated pneumonia probability')
plt.ylabel('Normalized predictive uncertainty')
plt.title('MCDS Decision Landscape')
plt.legend(fontsize=8, loc='best')
plt.tight_layout()
landscape_fig = FIG_DIR / 'mcds_decision_landscape_probability_uncertainty.png'
plt.savefig(landscape_fig, dpi=300, bbox_inches='tight')
plt.show()
log_message(f'Saved MCDS decision landscape to {landscape_fig}')






## 12. Safety-oriented decision audit

This table links selected actions to ground truth only for research auditing. In practice, the framework remains workflow decision support and not autonomous classification.





In [ ]:

audit = best_actions.copy()
audit['is_pneumonia'] = audit['y_true'].map({0:'Normal', 1:'Pneumonia'})
safety_table = pd.crosstab(audit['action'], audit['is_pneumonia'])
safety_path = TABLE_DIR / 'mcds_safety_audit_action_by_truth_rebalanced.csv'
safety_table.to_csv(safety_path)
print(safety_table)
log_message(f'Saved rebalanced MCDS safety audit table to {safety_path}')

# Identify potentially unsafe low-action pneumonia cases.
unsafe_mask = (audit['y_true'] == 1) & audit['action'].str.contains('Low-risk monitoring', regex=False)
unsafe_cases = audit.loc[unsafe_mask, [
    'case_id','pneumonia_probability_calibrated_mc','predictive_entropy',
    'predictive_uncertainty_normalized','mc_std','calibration_gap','action','mcds_score'
]]
unsafe_path = TABLE_DIR / 'potentially_unsafe_low_action_pneumonia_cases_rebalanced.csv'
unsafe_cases.to_csv(unsafe_path, index=False)
log_message(f'Potentially unsafe low-action pneumonia cases after rebalancing: {len(unsafe_cases)}')
log_message(f'Saved potentially unsafe case list to {unsafe_path}')

# A second safety check focuses on high-probability pneumonia cases.
high_prob_pneumonia = audit[(audit['y_true'] == 1) & (audit['pneumonia_probability_calibrated_mc'] >= 0.70)]
high_prob_low_action = high_prob_pneumonia[high_prob_pneumonia['action'].str.contains('Low-risk monitoring', regex=False)]
log_message(f'High-probability pneumonia cases assigned to low monitoring: {len(high_prob_low_action)}')

print(f'Potentially unsafe low-action pneumonia cases: {len(unsafe_cases)}')
print(f'High-probability pneumonia cases assigned to low monitoring: {len(high_prob_low_action)}')






## 13. Scenario analysis: low, baseline, and high epidemiological pressure

Scenario analysis demonstrates that the decision layer is context-sensitive. Higher epidemiological pressure should shift more cases toward review, additional evidence, or escalation.





In [ ]:

SCENARIOS = [
    {
        'name': 'low_prevalence_low_pressure',
        'seasonal_pressure': 0.10,
        'false_negative_cost': 0.95,
        'false_positive_cost': 0.55,
        'weights': {**SCENARIO['weights'], 'seasonal_pressure': 0.12, 'safety_barrier': 1.60}
    },
    SCENARIO,
    {
        'name': 'high_prevalence_high_pressure',
        'seasonal_pressure': 0.85,
        'false_negative_cost': 1.35,
        'false_positive_cost': 0.34,
        'weights': {**SCENARIO['weights'], 'seasonal_pressure': 0.36, 'confidence_reward': 0.82, 'safety_barrier': 1.95}
    },
    {
        'name': 'resource_constrained_setting',
        'seasonal_pressure': 0.45,
        'false_negative_cost': 1.20,
        'false_positive_cost': 0.50,
        'weights': {**SCENARIO['weights'], 'resource_burden': 0.46, 'action_complexity': 0.34, 'information_gain': 0.42}
    }
]

scenario_rows = []
scenario_selected = []
for sc in SCENARIOS:
    sdf = mcds_action_scores(mc_mean, mc_entropy, mc_std, cal_resid, sc)
    b = sdf.loc[sdf.groupby('case_id')['mcds_score'].idxmin()].reset_index(drop=True)
    scenario_selected.append(b.assign(scenario=sc['name']))
    dist = b['action'].value_counts().to_dict()
    for a in ACTIONS:
        scenario_rows.append({
            'scenario': sc['name'],
            'action': a,
            'count': int(dist.get(a, 0)),
            'percentage': 100 * int(dist.get(a, 0)) / len(b)
        })

scenario_df = pd.DataFrame(scenario_rows)
scenario_path = TABLE_DIR / 'mcds_scenario_action_comparison_rebalanced.csv'
scenario_df.to_csv(scenario_path, index=False)
log_message(f'Saved rebalanced MCDS scenario comparison to {scenario_path}')
print(scenario_df)

pivot = scenario_df.pivot(index='scenario', columns='action', values='count').fillna(0)
plt.figure(figsize=(10,5))
bottom = np.zeros(len(pivot))
for a in ACTIONS:
    vals = pivot[a].values if a in pivot.columns else np.zeros(len(pivot))
    plt.bar(pivot.index, vals, bottom=bottom, label=a)
    bottom += vals
plt.ylabel('Number of cases')
plt.title('Rebalanced MCDS Scenario-Sensitive Action Comparison')
plt.xticks(rotation=20, ha='right')
plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
scenario_fig = FIG_DIR / 'mcds_scenario_action_comparison_rebalanced.png'
plt.savefig(scenario_fig, dpi=300, bbox_inches='tight')
plt.show()
log_message(f'Saved rebalanced MCDS scenario action comparison figure to {scenario_fig}')

scenario_selected_df = pd.concat(scenario_selected, ignore_index=True)
scenario_selected_path = TABLE_DIR / 'mcds_scenario_selected_actions_rebalanced.csv'
scenario_selected_df.to_csv(scenario_selected_path, index=False)
log_message(f'Saved rebalanced scenario-selected actions to {scenario_selected_path}')






## MCDS ablation study

This cell quantifies the marginal contribution of the MCDS components. The full structured evaluation framework is compared with degraded variants that remove the soft safety barrier, remove the hard feasibility gate, remove the profile-distance term, remove the confidence reward, and replace MCDS with a plain probability-threshold classifier. The purpose is to show which terms affect safety auditing, action diversity, and threshold-sensitive routing.





In [ ]:
# =========================================================
# MCDS ablation study: full framework vs degraded variants
# =========================================================

def policy_entropy_from_actions(actions):
    counts = pd.Series(actions).value_counts()
    probs = counts / counts.sum()
    entropy = float(-(probs * np.log(probs)).sum())
    effective_actions = float(np.exp(entropy))
    return entropy, effective_actions

def summarize_policy(actions, y_true, prob=None, label='policy'):
    actions = np.asarray(actions, dtype=object)
    y_true = np.asarray(y_true).astype(int)
    unsafe = ((y_true == 1) & pd.Series(actions).str.contains('Low-risk monitoring', regex=False).to_numpy())
    fn_rate = float(unsafe.sum() / max((y_true == 1).sum(), 1))
    entropy, effective_actions = policy_entropy_from_actions(actions)
    out = {'variant': label, 'unsafe_low_action_pneumonia_cases': int(unsafe.sum()),
           'false_negative_rate_among_pneumonia': fn_rate, 'policy_entropy': entropy,
           'effective_number_of_actions': effective_actions, 'n_actions_used': int(pd.Series(actions).nunique())}
    if prob is not None:
        out['mean_probability'] = float(np.mean(prob))
    return out

def select_from_score_variant(score_source, variant_name):
    df = score_source.copy()
    w = SCENARIO['weights']
    profile_multiplier = 0.0 if variant_name == 'no_profile_distance' else 1.0
    safety_multiplier = 0.0 if variant_name == 'no_safety_barrier' else 1.0
    confidence_multiplier = 0.0 if variant_name == 'no_confidence_reward' else 1.0
    feasibility_multiplier = 0.0 if variant_name == 'no_feasibility_gate' else 1.0
    if 'safety_feasibility_penalty' not in df.columns:
        df['safety_feasibility_penalty'] = 0.0
    df['variant_score'] = (
        w['risk_cost'] * df['risk_cost_component']
        + w['uncertainty'] * df['uncertainty_component']
        + w['calibration'] * df['calibration_gap']
        + w['seasonal_pressure'] * df.get('seasonal_pressure_component', df['seasonal_pressure'])
        + w['resource_burden'] * df['resource_burden']
        + w['action_complexity'] * df['action_complexity']
        + profile_multiplier * w['profile_distance'] * df['profile_distance_component']
        + safety_multiplier * w['safety_barrier'] * df['safety_barrier_component']
        + feasibility_multiplier * df['safety_feasibility_penalty']
        - w['information_gain'] * df['information_gain'] * df['information_reward_component']
        - confidence_multiplier * w['confidence_reward'] * df['confidence_reward_component']
    )
    return df.loc[df.groupby('case_id')['variant_score'].idxmin()].reset_index(drop=True)

ablation_rows = []
ablation_rows.append(summarize_policy(best_actions['action'], y_test, prob=best_actions['pneumonia_probability_calibrated_mc'], label='full_mcds'))
for variant in ['no_safety_barrier', 'no_feasibility_gate', 'no_profile_distance', 'no_confidence_reward']:
    selected_variant = select_from_score_variant(score_df, variant)
    selected_variant.to_csv(TABLE_DIR / f'mcds_selected_actions_{variant}.csv', index=False)
    ablation_rows.append(summarize_policy(selected_variant['action'], y_test, prob=selected_variant['pneumonia_probability_calibrated_mc'], label=variant))
plain_actions = np.where(mc_mean >= CAL_OPT_THR, 'Treat as likely pneumonia with clinician confirmation', 'Low-risk monitoring / outpatient follow-up')
ablation_rows.append(summarize_policy(plain_actions, y_test, prob=mc_mean, label='plain_threshold_classifier'))
ablation_df = pd.DataFrame(ablation_rows)
ablation_path = TABLE_DIR / 'mcds_ablation_comparison.csv'
ablation_df.to_csv(ablation_path, index=False)
print(ablation_df.round(4))
log_message(f'Saved MCDS ablation comparison to {ablation_path}')

plt.figure(figsize=(8, 4.2))
x = np.arange(len(ablation_df))
plt.bar(x - 0.18, ablation_df['unsafe_low_action_pneumonia_cases'], width=0.36, label='Unsafe cases')
plt.bar(x + 0.18, ablation_df['false_negative_rate_among_pneumonia'], width=0.36, label='FN rate')
plt.xticks(x, ablation_df['variant'], rotation=25, ha='right')
plt.ylabel('Count / rate')
plt.title('MCDS Ablation: Safety and False-Negative Behavior')
plt.legend()
plt.tight_layout()
ablation_fig = FIG_DIR / 'ablation_bar_chart.png'
plt.savefig(ablation_fig, dpi=300, bbox_inches='tight')
plt.show()
log_message(f'Saved ablation bar chart to {ablation_fig}')







## Structured scenario sweep

This cell evaluates the configurability of the MCDS evaluation layer across workflow-relevant risk profiles. Pressure levels are crossed with false-negative and false-positive cost settings to test whether the routing behavior changes coherently under different benchmark conditions.





In [ ]:
# =========================================================
# Structured workflow scenario sweep: 3 x 3 grid
# =========================================================

pressure_levels = {'low_pressure': 0.15, 'moderate_pressure': 0.40, 'high_pressure': 0.70}
cost_profiles = {
    'primary_care': {'false_negative_cost': 0.80, 'false_positive_cost': 0.40},
    'emergency': {'false_negative_cost': 1.50, 'false_positive_cost': 0.40},
    'outbreak': {'false_negative_cost': 2.00, 'false_positive_cost': 0.30}
}
scenario_sweep_rows = []
scenario_entropy_grid = pd.DataFrame(index=list(cost_profiles.keys()), columns=list(pressure_levels.keys()), dtype=float)
for profile_name, costs in cost_profiles.items():
    for pressure_name, pressure_value in pressure_levels.items():
        sc = {'name': f'{profile_name}_{pressure_name}', 'seasonal_pressure': pressure_value,
              'false_negative_cost': costs['false_negative_cost'], 'false_positive_cost': costs['false_positive_cost'],
              'weights': SCENARIO['weights'].copy()}
        sdf = mcds_action_scores(mc_mean, mc_entropy, mc_std, cal_resid, sc)
        b = sdf.loc[sdf.groupby('case_id')['mcds_score'].idxmin()].reset_index(drop=True)
        entropy, eff_actions = policy_entropy_from_actions(b['action'])
        unsafe_count = int(((b['y_true'] == 1) & b['action'].str.contains('Low-risk monitoring', regex=False)).sum())
        scenario_entropy_grid.loc[profile_name, pressure_name] = entropy
        dist = b['action'].value_counts().to_dict()
        for action in ACTIONS:
            scenario_sweep_rows.append({'cost_profile': profile_name, 'pressure_level': pressure_name, 'scenario': sc['name'],
                                        'action': action, 'count': int(dist.get(action, 0)),
                                        'percentage': 100 * int(dist.get(action, 0)) / len(b),
                                        'policy_entropy': entropy, 'effective_number_of_actions': eff_actions,
                                        'unsafe_low_action_pneumonia_cases': unsafe_count,
                                        'false_negative_cost': costs['false_negative_cost'], 'false_positive_cost': costs['false_positive_cost'],
                                        'seasonal_pressure': pressure_value})
scenario_sweep_df = pd.DataFrame(scenario_sweep_rows)
scenario_sweep_path = TABLE_DIR / 'scenario_sweep_action_distributions.csv'
scenario_sweep_df.to_csv(scenario_sweep_path, index=False)
scenario_entropy_path = TABLE_DIR / 'scenario_policy_entropy_grid.csv'
scenario_entropy_grid.to_csv(scenario_entropy_path)
print(scenario_entropy_grid.round(4))
log_message(f'Saved scenario sweep action distributions to {scenario_sweep_path}')
log_message(f'Saved scenario policy entropy grid to {scenario_entropy_path}')
plt.figure(figsize=(6.5, 4.8))
plt.imshow(scenario_entropy_grid.to_numpy(dtype=float), aspect='auto')
plt.xticks(range(len(scenario_entropy_grid.columns)), scenario_entropy_grid.columns, rotation=20, ha='right')
plt.yticks(range(len(scenario_entropy_grid.index)), scenario_entropy_grid.index)
plt.colorbar(label='Policy entropy')
plt.title('Scenario Sweep: MCDS Policy Entropy')
plt.tight_layout()
scenario_heatmap = FIG_DIR / 'scenario_policy_entropy_heatmap.png'
plt.savefig(scenario_heatmap, dpi=300, bbox_inches='tight')
plt.show()
log_message(f'Saved scenario policy entropy heatmap to {scenario_heatmap}')






## 14. Threshold sensitivity analysis

This cell compares simple probability-threshold policies with structured MCDS routing. The purpose is to show how threshold choice changes sensitivity, specificity, false negatives, and balanced accuracy.





In [ ]:
sens_rows = []
for t in np.linspace(0.1, 0.9, 17):
    pred = (mc_mean >= t).astype(int)
    cm = confusion_matrix(y_test, pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    sens_rows.append({
        'threshold': float(t),
        'accuracy': accuracy_score(y_test, pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall_sensitivity': recall_score(y_test, pred, zero_division=0),
        'specificity': tn/(tn+fp) if (tn+fp)>0 else np.nan,
        'false_negatives': int(fn),
        'false_positives': int(fp),
        'tp': int(tp),
        'tn': int(tn)
    })

sens_df = pd.DataFrame(sens_rows)
sens_path = TABLE_DIR / 'probability_threshold_sensitivity_vs_mcds.csv'
sens_df.to_csv(sens_path, index=False)
print(sens_df.round(4))
log_message(f'Saved probability threshold sensitivity analysis to {sens_path}')

plt.figure(figsize=(6,4))
plt.plot(sens_df['threshold'], sens_df['recall_sensitivity'], marker='o', label='Sensitivity')
plt.plot(sens_df['threshold'], sens_df['specificity'], marker='o', label='Specificity')
plt.plot(sens_df['threshold'], sens_df['balanced_accuracy'], marker='o', label='Balanced accuracy')
plt.xlabel('Probability threshold')
plt.ylabel('Metric')
plt.title('Threshold Sensitivity')
plt.legend()
plt.tight_layout()
th_fig = FIG_DIR / 'probability_threshold_sensitivity.png'
plt.savefig(th_fig, dpi=300, bbox_inches='tight')
plt.show()
log_message(f'Saved threshold sensitivity figure to {th_fig}')





## 15. Generate manuscript-oriented interpretation summary

This final cell writes a concise interpretation that can guide the Array manuscript revision, emphasizing reproducibility, calibration, threshold optimization, uncertainty diagnostics, and explainable evaluation.





In [ ]:
# =========================================================
# Final Interpretation Summary and Output Manifest
# =========================================================

import numpy as np
import pandas as pd

def _safe_float(x, default=np.nan):
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default

def _metric(metrics_dict, key, default=np.nan):
    if isinstance(metrics_dict, dict) and key in metrics_dict:
        return _safe_float(metrics_dict[key], default)
    return default

def _get_existing_value(names, default=np.nan):
    for name in names:
        if name in globals():
            return globals()[name]
    return default

def _find_first_existing_file(paths):
    for p in paths:
        try:
            if p.exists():
                return p
        except Exception:
            pass
    return None

# ---------------------------------------------------------
# Recover ECE values robustly
# ---------------------------------------------------------

raw_ece_value = _safe_float(
    _get_existing_value(["RAW_ECE_REFERENCE", "raw_ece", "raw_ece_test", "raw_test_ece"], np.nan)
)

cal_ece_value = _safe_float(
    _get_existing_value(["FINAL_CAL_ECE", "CAL_ECE_REFERENCE", "cal_ece", "cal_ece_test", "cal_test_ece"], np.nan)
)

df_cal = None
if "calibration_comparison_df" in globals() and isinstance(calibration_comparison_df, pd.DataFrame):
    df_cal = calibration_comparison_df.copy()
elif "TABLE_DIR" in globals():
    cal_csv = _find_first_existing_file([
        TABLE_DIR / "calibration_method_comparison.csv",
        TABLE_DIR / "calibration_method_comparison_guarded.csv",
        TABLE_DIR / "extended_calibration_diagnostics.csv",
    ])
    if cal_csv is not None:
        try:
            df_cal = pd.read_csv(cal_csv)
        except Exception:
            df_cal = None

if df_cal is not None and isinstance(df_cal, pd.DataFrame) and len(df_cal) > 0:
    df_cal = df_cal.copy()
    df_cal.columns = [str(c).strip().lower() for c in df_cal.columns]
    method_col = "method" if "method" in df_cal.columns else None
    ece_col = next((c for c in ["test_ece", "ece_test", "ece", "calibrated_ece", "cal_ece"] if c in df_cal.columns), None)

    if method_col and ece_col:
        method_lower = df_cal[method_col].astype(str).str.lower()

        if np.isnan(raw_ece_value):
            raw_rows = df_cal[method_lower.isin(["raw", "raw_uncalibrated", "uncalibrated", "raw_model", "baseline"])]
            if len(raw_rows):
                raw_ece_value = _safe_float(raw_rows.iloc[0][ece_col])

        if "FINAL_CALIBRATION_METHOD" in globals():
            selected_method = str(FINAL_CALIBRATION_METHOD).lower()
        elif "CALIBRATION_METHOD" in globals():
            selected_method = str(CALIBRATION_METHOD).lower()
        else:
            selected_method = None

        if selected_method is not None and np.isnan(cal_ece_value):
            selected_rows = df_cal[method_lower.eq(selected_method)]
            if len(selected_rows):
                cal_ece_value = _safe_float(selected_rows.iloc[0][ece_col])

        if np.isnan(cal_ece_value):
            non_raw = df_cal[~method_lower.isin(["raw", "raw_uncalibrated", "uncalibrated", "raw_model", "baseline"])]
            if len(non_raw):
                cal_ece_value = _safe_float(non_raw.sort_values(ece_col).iloc[0][ece_col])

# Last-resort recomputation if arrays and calibration functions exist
if (np.isnan(raw_ece_value) or np.isnan(cal_ece_value)) and "calibration_metrics" in globals():
    try:
        if np.isnan(raw_ece_value) and "y_test" in globals() and "p_test_raw" in globals():
            raw_tmp, _ = calibration_metrics(y_test, p_test_raw, n_bins=10)
            raw_ece_value = raw_tmp.get('ece', np.nan)
        if np.isnan(cal_ece_value) and "y_test" in globals() and "p_test_cal" in globals():
            cal_tmp, _ = calibration_metrics(y_test, p_test_cal, n_bins=10)
            cal_ece_value = cal_tmp.get('ece', np.nan)
    except Exception:
        pass

if (np.isnan(raw_ece_value) or np.isnan(cal_ece_value)) and "expected_calibration_error" in globals():
    try:
        if np.isnan(raw_ece_value) and "y_test" in globals() and "p_test_raw" in globals():
            raw_ece_value, _ = expected_calibration_error(y_test, p_test_raw, n_bins=10)
        if np.isnan(cal_ece_value) and "y_test" in globals() and "p_test_cal" in globals():
            cal_ece_value, _ = expected_calibration_error(y_test, p_test_cal, n_bins=10)
    except Exception:
        pass

if np.isnan(raw_ece_value) or np.isnan(cal_ece_value):
    raise ValueError('Final summary could not recover finite raw/calibrated ECE values. Run the calibration diagnostics cell first.')

raw_ece_value = _safe_float(raw_ece_value)
cal_ece_value = _safe_float(cal_ece_value)

# ---------------------------------------------------------
# Recover calibration, uncertainty, and policy metadata
# ---------------------------------------------------------

if "FINAL_CALIBRATION_METHOD" in globals():
    calibration_method_text = str(FINAL_CALIBRATION_METHOD)
elif "CALIBRATION_METHOD" in globals():
    calibration_method_text = str(CALIBRATION_METHOD)
else:
    calibration_method_text = "reported in calibration_method_comparison.csv"

validation_selected_text = str(VALIDATION_SELECTED_CALIBRATION_METHOD) if "VALIDATION_SELECTED_CALIBRATION_METHOD" in globals() else None
temperature_value = _safe_float(_get_existing_value(["TEMP", "best_temperature", "temperature"], np.nan))

mc_entropy_mean = _safe_float(np.mean(mc_entropy)) if "mc_entropy" in globals() else np.nan
mc_std_mean = _safe_float(np.mean(mc_std)) if "mc_std" in globals() else np.nan

policy_entropy_value = _safe_float(_get_existing_value(["policy_entropy"], np.nan))
policy_effective_value = _safe_float(_get_existing_value(["policy_effective_actions"], np.nan))
if np.isnan(policy_effective_value) and not np.isnan(policy_entropy_value):
    policy_effective_value = float(np.exp(policy_entropy_value))

if "unsafe_cases" in globals():
    try:
        unsafe_count = len(unsafe_cases)
    except Exception:
        unsafe_count = np.nan
else:
    unsafe_count = np.nan

# ---------------------------------------------------------
# Build summary
# ---------------------------------------------------------

summary_lines = []
summary_lines.append("Array Pneumonia Calibration Pipeline — Interpretation Summary")
summary_lines.append("=" * 72)
summary_lines.append(f'Raw ROC-AUC: {_metric(raw_metrics, "roc_auc"):.4f}')
summary_lines.append(f'Raw balanced accuracy: {_metric(raw_metrics, "balanced_accuracy"):.4f}')
summary_lines.append(f'Calibrated ROC-AUC: {_metric(cal_metrics, "roc_auc"):.4f}')
summary_lines.append(f'Calibrated balanced accuracy: {_metric(cal_metrics, "balanced_accuracy"):.4f}')
summary_lines.append(f"Raw ECE: {raw_ece_value:.4f}")
summary_lines.append(f"Calibrated ECE: {cal_ece_value:.4f}")
summary_lines.append(f"Calibration method: {calibration_method_text}")
summary_lines.append("Calibration reporting note: temperature scaling is used as the manuscript-facing calibrated model because it preserves probability ranking; non-monotonic/flexible calibrators remain comparison baselines.")

if validation_selected_text is not None:
    summary_lines.append(f"Validation-selected calibration method: {validation_selected_text}")
if not np.isnan(temperature_value):
    summary_lines.append(f"Temperature parameter: {temperature_value:.4f}")

summary_lines.append(f"Mean MC-dropout entropy: {mc_entropy_mean:.4f}")
summary_lines.append(f"Mean MC-dropout std: {mc_std_mean:.4f}")
summary_lines.append(f"Policy entropy: {policy_entropy_value:.4f}")
summary_lines.append(f"Effective number of selected actions: {policy_effective_value:.2f}")
summary_lines.append(f"Potentially unsafe low-action pneumonia cases: {unsafe_count}")

if "SAFETY_FEASIBILITY" in globals():
    summary_lines.append(
        "Low-risk feasibility gate: monitoring is infeasible when "
        f'pneumonia probability > {SAFETY_FEASIBILITY["low_monitoring_max_probability"]:.2f} '
        f'or normalized uncertainty > {SAFETY_FEASIBILITY["low_monitoring_max_uncertainty"]:.2f}'
    )

summary_lines.append("")
summary_lines.append("Main interpretation:")
summary_lines.append(
    "The notebook separates pneumonia probability estimation from review action selection. "
    "The predictive model provides evidence, while MC-dropout uncertainty, calibration reliability, "
    "workflow cost asymmetry, epidemiological pressure, action burden, confidence reward, "
    "action-specific suitability, and a safety-feasibility gate are integrated into a constrained "
    "MCDS structured workflow evaluation score."
)
summary_lines.append("")
summary_lines.append("Formal decision principle:")
summary_lines.append(
    "For each case, workflow-infeasible low-risk actions are first excluded by the safety-feasibility gate. "
    "The selected action then minimizes the MCDS functional over the remaining candidate actions. "
    "This hybrid design combines a non-negotiable decision-support safety floor with flexible structured evaluation."
)
summary_lines.append("")
summary_lines.append("Baseline constrained MCDS action distribution:")

if "action_dist" in globals() and isinstance(action_dist, pd.DataFrame):
    for _, row in action_dist.iterrows():
        summary_lines.append(f'- {row["action"]}: {int(row["count"])} cases ({float(row["percentage"]):.2f}%)')
else:
    summary_lines.append("- Action distribution was not available in memory.")

summary_lines.append("")
summary_lines.append("Recommended manuscript framing:")
summary_lines.append(
    "The manuscript should emphasize a reproducible calibration-aware AI evaluation framework: the model first estimates pneumonia risk, "
    "then constrained MCDS structures whether the evidence is sufficient for monitoring, treatment confirmation, "
    "additional evidence, clinician review, or escalation."
)
summary_lines.append("")
summary_lines.append("Critical caution:")
summary_lines.append(
    "This framework is designed for decision support only. It should not be described as autonomous diagnosis "
    "or replacement of clinician judgment."
)

with open(INTERPRET_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

print("\n".join(summary_lines))

if "log_message" in globals():
    log_message(f"Saved final interpretation summary to {INTERPRET_PATH}")
else:
    print(f"Saved final interpretation summary to {INTERPRET_PATH}")

manifest = []
for folder in [FIG_DIR, TABLE_DIR, OUTPUT_DIR]:
    if folder.exists():
        for pth in sorted(folder.glob("*")):
            manifest.append({"folder": folder.name, "file": pth.name, "path": str(pth)})

manifest_df = pd.DataFrame(manifest)
manifest_path = OUTPUT_DIR / "generated_files_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

if "log_message" in globals():
    log_message(f"Saved generated files manifest to {manifest_path}")
else:
    print(f"Saved generated files manifest to {manifest_path}")

manifest_df.head(20)







## 16. External transferability validation on a second chest X-ray dataset

This section adds a reduced second-dataset experiment using the Kaggle/Kermany-style chest X-ray folder structure:

```text
Chest_Xray/
├── train/
│   ├── NORMAL/
│   └── PNEUMONIA/
└── test/
    ├── NORMAL/
    └── PNEUMONIA/
```

The purpose is reviewer-oriented: to show that the MCDS decision layer can be transferred beyond PneumoniaMNIST without redesigning the framework. The experiment uses a frozen pretrained MobileNetV2 backbone with dropout-enabled inference, temperature calibration, MC-dropout uncertainty, and the same MCDS action-selection logic.






In [ ]:
# =========================================================
# 16.1 External dataset path and structure verification
# =========================================================

EXTERNAL_DATASET_PATH = Path('/content/drive/MyDrive/Datasets/Pneumonia/Chest_Xray')
EXTERNAL_TRAIN_DIR = EXTERNAL_DATASET_PATH / 'train'
EXTERNAL_TEST_DIR  = EXTERNAL_DATASET_PATH / 'test'

RUN_EXTERNAL_VALIDATION = (
    EXTERNAL_TRAIN_DIR.exists()
    and EXTERNAL_TEST_DIR.exists()
    and (EXTERNAL_TRAIN_DIR / 'NORMAL').exists()
    and (EXTERNAL_TRAIN_DIR / 'PNEUMONIA').exists()
    and (EXTERNAL_TEST_DIR / 'NORMAL').exists()
    and (EXTERNAL_TEST_DIR / 'PNEUMONIA').exists()
)

print('External dataset path:', EXTERNAL_DATASET_PATH)
print('External validation enabled:', RUN_EXTERNAL_VALIDATION)

if RUN_EXTERNAL_VALIDATION:
    ext_counts = []
    for split_name, split_dir in [('train', EXTERNAL_TRAIN_DIR), ('test', EXTERNAL_TEST_DIR)]:
        for cls in ['NORMAL', 'PNEUMONIA']:
            cls_dir = split_dir / cls
            files = [f for f in cls_dir.iterdir() if f.is_file()]
            ext_counts.append({'split': split_name, 'class': cls, 'count': len(files)})
            print(f'{split_name:>5} | {cls:<9}: {len(files)} images')

    ext_counts_df = pd.DataFrame(ext_counts)
    ext_counts_path = TABLE_DIR / 'external_dataset_image_counts.csv'
    ext_counts_df.to_csv(ext_counts_path, index=False)
    log_message(f'Saved external dataset image counts to {ext_counts_path}')
else:
    log_message('External dataset not found. Skipping second-dataset validation cells.')






In [ ]:
# =========================================================
# 16.2 Build TensorFlow datasets for external validation
# =========================================================

if RUN_EXTERNAL_VALIDATION:
    EXT_IMG_SIZE = 128
    EXT_BATCH_SIZE = 32
    EXT_VAL_SPLIT = 0.15

    ext_train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        EXTERNAL_TRAIN_DIR,
        labels='inferred',
        label_mode='binary',
        color_mode='rgb',
        image_size=(EXT_IMG_SIZE, EXT_IMG_SIZE),
        batch_size=EXT_BATCH_SIZE,
        validation_split=EXT_VAL_SPLIT,
        subset='training',
        seed=RANDOM_STATE,
        shuffle=True
    )

    ext_val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        EXTERNAL_TRAIN_DIR,
        labels='inferred',
        label_mode='binary',
        color_mode='rgb',
        image_size=(EXT_IMG_SIZE, EXT_IMG_SIZE),
        batch_size=EXT_BATCH_SIZE,
        validation_split=EXT_VAL_SPLIT,
        subset='validation',
        seed=RANDOM_STATE,
        shuffle=False
    )

    ext_test_ds = tf.keras.preprocessing.image_dataset_from_directory(
        EXTERNAL_TEST_DIR,
        labels='inferred',
        label_mode='binary',
        color_mode='rgb',
        image_size=(EXT_IMG_SIZE, EXT_IMG_SIZE),
        batch_size=EXT_BATCH_SIZE,
        shuffle=False
    )

    EXT_CLASS_NAMES = ext_train_ds.class_names
    print('External class names:', EXT_CLASS_NAMES)

    # Prefetching improves Colab throughput without forcing all images into memory.
    AUTOTUNE = tf.data.AUTOTUNE
    ext_train_ds = ext_train_ds.prefetch(AUTOTUNE)
    ext_val_ds   = ext_val_ds.prefetch(AUTOTUNE)
    ext_test_ds  = ext_test_ds.prefetch(AUTOTUNE)

    def dataset_labels(ds):
        labels = []
        for _, yb in ds:
            labels.append(yb.numpy().reshape(-1))
        return np.concatenate(labels).astype(int)

    ext_y_val = dataset_labels(ext_val_ds)
    ext_y_test = dataset_labels(ext_test_ds)

    # Class weights are computed from the full external training directory.
    train_counts = {cls: len([f for f in (EXTERNAL_TRAIN_DIR / cls).iterdir() if f.is_file()]) for cls in EXT_CLASS_NAMES}
    total_train = sum(train_counts.values())
    ext_class_weight = {
        idx: total_train / (len(EXT_CLASS_NAMES) * max(train_counts[cls], 1))
        for idx, cls in enumerate(EXT_CLASS_NAMES)
    }
    print('External train counts:', train_counts)
    print('External class weights:', ext_class_weight)






In [ ]:
# =========================================================
# 16.3 Lightweight transfer model with MC-dropout head
# =========================================================

if RUN_EXTERNAL_VALIDATION:
    def build_external_transfer_model(input_shape=(128, 128, 3), dropout_rate=0.35):
        inputs = keras.Input(shape=input_shape)
        x = keras.applications.mobilenet_v2.preprocess_input(inputs)
        base = keras.applications.MobileNetV2(
            include_top=False,
            weights='imagenet',
            input_shape=input_shape
        )
        base.trainable = False

        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(dropout_rate)(x)
        x = layers.Dense(128, activation='relu')(x)
        x = layers.Dropout(dropout_rate)(x)
        outputs = layers.Dense(1, activation='sigmoid')(x)

        model_ext = keras.Model(inputs, outputs, name='external_transfer_mobilenetv2_mc_dropout')
        model_ext.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-4),
            loss='binary_crossentropy',
            metrics=[
                keras.metrics.BinaryAccuracy(name='accuracy'),
                keras.metrics.AUC(name='auc')
            ]
        )
        return model_ext

    external_model = build_external_transfer_model(
        input_shape=(EXT_IMG_SIZE, EXT_IMG_SIZE, 3),
        dropout_rate=0.35
    )

    external_model.summary()

    ext_callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_auc',
            mode='max',
            patience=3,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc',
            mode='max',
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )
    ]

    log_message('Starting lightweight external transfer training.')
    ext_history = external_model.fit(
        ext_train_ds,
        validation_data=ext_val_ds,
        epochs=8,
        class_weight=ext_class_weight,
        callbacks=ext_callbacks,
        verbose=1
    )
    log_message('Finished lightweight external transfer training.')

    # Save training curves for the external experiment.
    ext_hist_df = pd.DataFrame(ext_history.history)
    ext_hist_path = TABLE_DIR / 'external_transfer_training_history.csv'
    ext_hist_df.to_csv(ext_hist_path, index=False)

    plt.figure(figsize=(6, 4))
    plt.plot(ext_hist_df['loss'], marker='o', label='train loss')
    plt.plot(ext_hist_df['val_loss'], marker='o', label='val loss')
    plt.xlabel('Epoch')
    plt.ylabel('Binary cross-entropy')
    plt.title('External Transfer Training Loss')
    plt.legend()
    plt.tight_layout()
    ext_loss_fig = FIG_DIR / 'external_transfer_training_loss.png'
    plt.savefig(ext_loss_fig, dpi=300, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.plot(ext_hist_df['auc'], marker='o', label='train AUC')
    plt.plot(ext_hist_df['val_auc'], marker='o', label='val AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.title('External Transfer Training AUC')
    plt.legend()
    plt.tight_layout()
    ext_auc_fig = FIG_DIR / 'external_transfer_training_auc.png'
    plt.savefig(ext_auc_fig, dpi=300, bbox_inches='tight')
    plt.show()

    log_message(f'Saved external transfer training history to {ext_hist_path}')






In [ ]:
# =========================================================
# 16.4 External predictive evaluation and temperature calibration
# =========================================================

if RUN_EXTERNAL_VALIDATION:
    ext_p_val_raw = external_model.predict(ext_val_ds, verbose=0).reshape(-1)
    ext_p_test_raw = external_model.predict(ext_test_ds, verbose=0).reshape(-1)

    ext_threshold_grid = np.linspace(0.05, 0.95, 181)
    ext_val_baccs = [
        balanced_accuracy_score(ext_y_val, (ext_p_val_raw >= t).astype(int))
        for t in ext_threshold_grid
    ]
    EXT_RAW_OPT_THR = float(ext_threshold_grid[int(np.argmax(ext_val_baccs))])

    ext_raw_pred = (ext_p_test_raw >= EXT_RAW_OPT_THR).astype(int)
    ext_raw_metrics = {
        'dataset': 'Chest_Xray_external',
        'model': 'MobileNetV2_frozen_dropout_head',
        'threshold': EXT_RAW_OPT_THR,
        'accuracy': accuracy_score(ext_y_test, ext_raw_pred),
        'balanced_accuracy': balanced_accuracy_score(ext_y_test, ext_raw_pred),
        'precision': precision_score(ext_y_test, ext_raw_pred, zero_division=0),
        'recall': recall_score(ext_y_test, ext_raw_pred, zero_division=0),
        'f1': f1_score(ext_y_test, ext_raw_pred, zero_division=0),
        'roc_auc': roc_auc_score(ext_y_test, ext_p_test_raw),
        'average_precision': average_precision_score(ext_y_test, ext_p_test_raw),
        'brier_score': brier_score_loss(ext_y_test, ext_p_test_raw)
    }

    ext_val_logits = prob_to_logit(ext_p_val_raw)
    ext_test_logits = prob_to_logit(ext_p_test_raw)
    ext_temp_res = minimize_scalar(
        lambda T: nll_temperature(T, ext_val_logits, ext_y_val),
        bounds=(0.2, 10.0),
        method='bounded'
    )
    EXT_TEMP = float(ext_temp_res.x)

    ext_p_val_cal = sigmoid(ext_val_logits / EXT_TEMP)
    ext_p_test_cal = sigmoid(ext_test_logits / EXT_TEMP)

    ext_cal_baccs = [
        balanced_accuracy_score(ext_y_val, (ext_p_val_cal >= t).astype(int))
        for t in ext_threshold_grid
    ]
    EXT_CAL_OPT_THR = float(ext_threshold_grid[int(np.argmax(ext_cal_baccs))])

    ext_cal_pred = (ext_p_test_cal >= EXT_CAL_OPT_THR).astype(int)
    ext_cal_metrics = {
        'dataset': 'Chest_Xray_external',
        'model': 'MobileNetV2_frozen_dropout_head_temperature_scaled',
        'temperature': EXT_TEMP,
        'threshold': EXT_CAL_OPT_THR,
        'accuracy': accuracy_score(ext_y_test, ext_cal_pred),
        'balanced_accuracy': balanced_accuracy_score(ext_y_test, ext_cal_pred),
        'precision': precision_score(ext_y_test, ext_cal_pred, zero_division=0),
        'recall': recall_score(ext_y_test, ext_cal_pred, zero_division=0),
        'f1': f1_score(ext_y_test, ext_cal_pred, zero_division=0),
        'roc_auc': roc_auc_score(ext_y_test, ext_p_test_cal),
        'average_precision': average_precision_score(ext_y_test, ext_p_test_cal),
        'brier_score': brier_score_loss(ext_y_test, ext_p_test_cal)
    }

    ext_metrics_df = pd.DataFrame([ext_raw_metrics, ext_cal_metrics])
    ext_metrics_path = TABLE_DIR / 'external_chestxray_predictive_metrics.csv'
    ext_metrics_df.to_csv(ext_metrics_path, index=False)
    print(ext_metrics_df.round(4))
    log_message(f'External optimized temperature: {EXT_TEMP:.4f}')
    log_message(f'Saved external predictive metrics to {ext_metrics_path}')

    # Confusion matrix after calibration.
    ext_cm = confusion_matrix(ext_y_test, ext_cal_pred, labels=[0, 1])
    plt.figure(figsize=(4, 4))
    plt.imshow(ext_cm)
    plt.title('External Calibrated Confusion Matrix')
    plt.xticks([0, 1], ['Normal', 'Pneumonia'])
    plt.yticks([0, 1], ['Normal', 'Pneumonia'])
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(ext_cm[i, j]), ha='center', va='center')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    ext_cm_fig = FIG_DIR / 'external_chestxray_calibrated_confusion_matrix.png'
    plt.savefig(ext_cm_fig, dpi=300, bbox_inches='tight')
    plt.show()
    log_message(f'Saved external calibrated confusion matrix to {ext_cm_fig}')






In [ ]:
# =========================================================
# 16.5 External calibration diagnostics and MC-dropout uncertainty
# =========================================================

if RUN_EXTERNAL_VALIDATION:

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.calibration import calibration_curve

    # -----------------------------------------------------
    # Helper functions (local + self-contained)
    # -----------------------------------------------------

    def sigmoid(z):
        z = np.asarray(z, dtype=float)
        return 1.0 / (1.0 + np.exp(-z))

    def prob_to_logit(p, eps=1e-7):
        p = np.asarray(p, dtype=float)
        p = np.clip(p, eps, 1.0 - eps)
        return np.log(p / (1.0 - p))

    def entropy_binary(p, eps=1e-7):
        p = np.asarray(p, dtype=float)
        p = np.clip(p, eps, 1.0 - eps)
        return -(p * np.log2(p) + (1.0 - p) * np.log2(1.0 - p))

    def expected_calibration_error(y_true, y_prob, n_bins=10):
        # Compute ECE and return bin-level statistics.
        y_true = np.asarray(y_true).reshape(-1).astype(int)
        y_prob = np.asarray(y_prob).reshape(-1).astype(float)
        y_prob = np.clip(y_prob, 0.0, 1.0)
        edges = np.linspace(0.0, 1.0, n_bins + 1)
        ece = 0.0
        rows = []
        for b in range(n_bins):
            lo, hi = edges[b], edges[b + 1]
            if b == n_bins - 1:
                mask = (y_prob >= lo) & (y_prob <= hi)
            else:
                mask = (y_prob >= lo) & (y_prob < hi)
            count = int(mask.sum())
            if count:
                conf = float(y_prob[mask].mean())
                obs = float(y_true[mask].mean())
                gap = abs(conf - obs)
                ece += (count / len(y_prob)) * gap
            else:
                conf, obs, gap = np.nan, np.nan, np.nan
            rows.append({
                'bin': b,
                'lo': lo,
                'hi': hi,
                'count': count,
                'mean_confidence': conf,
                'observed_frequency': obs,
                'gap': gap,
                'abs_gap': gap
            })
        return float(ece), pd.DataFrame(rows)

    def calibration_residual(probabilities, bins_df):
        # Map each probability to the absolute calibration gap of its bin.
        probabilities = np.asarray(probabilities).reshape(-1).astype(float)
        residuals = np.zeros_like(probabilities, dtype=float)
        for _, row in bins_df.iterrows():
            lo = float(row.get('lo', row.get('bin_left', 0.0)))
            hi = float(row.get('hi', row.get('bin_right', 1.0)))
            gap = row.get('gap', row.get('abs_gap', 0.0))
            if pd.isna(gap):
                gap = 0.0
            if int(row['bin']) == int(bins_df['bin'].max()):
                mask = (probabilities >= lo) & (probabilities <= hi)
            else:
                mask = (probabilities >= lo) & (probabilities < hi)
            residuals[mask] = float(gap)
        return residuals

    ext_val_ece, ext_val_bins = expected_calibration_error(ext_y_val, ext_p_val_cal, n_bins=10)
    ext_test_ece, ext_test_bins = expected_calibration_error(ext_y_test, ext_p_test_cal, n_bins=10)

    ext_val_bins.to_csv(TABLE_DIR / 'external_validation_calibration_bins.csv', index=False)
    ext_test_bins.to_csv(TABLE_DIR / 'external_test_calibration_bins.csv', index=False)

    log_message(f'External validation ECE after calibration: {ext_val_ece:.4f}')
    log_message(f'External test ECE after calibration: {ext_test_ece:.4f}')

    plt.figure(figsize=(5.5, 5))
    for prob, name in [(ext_p_test_raw, 'Raw'), (ext_p_test_cal, 'Calibrated')]:
        frac_pos, mean_pred = calibration_curve(ext_y_test, prob, n_bins=10, strategy='uniform')
        plt.plot(mean_pred, frac_pos, marker='o', label=name)
    plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect calibration')
    plt.xlabel('Mean predicted probability')
    plt.ylabel('Observed pneumonia frequency')
    plt.title('External Reliability Curve')
    plt.legend()
    plt.tight_layout()
    ext_rel_fig = FIG_DIR / 'external_reliability_curve_raw_vs_calibrated.png'
    plt.savefig(ext_rel_fig, dpi=300, bbox_inches='tight')
    plt.show()
    log_message(f'Saved external reliability curve to {ext_rel_fig}')

    def mc_dropout_predict_dataset(model, ds, n_passes=30):
        all_passes = []
        for i in range(n_passes):
            if (i + 1) % 10 == 0:
                print(f'External MC pass {i + 1}/{n_passes}')
            pass_preds = []
            for xb, _ in ds:
                pass_preds.append(model(xb, training=True).numpy().reshape(-1))
            all_passes.append(np.concatenate(pass_preds))
        return np.vstack(all_passes)

    ext_mc_raw = mc_dropout_predict_dataset(external_model, ext_test_ds, n_passes=30)

    if 'EXT_TEMP' in globals():
        ext_temp_used = EXT_TEMP
    else:
        ext_temp_used = 1.0
        log_message('EXT_TEMP not found; using 1.0 for external MC calibration.')

    ext_mc_logits = prob_to_logit(ext_mc_raw)
    ext_mc_cal = sigmoid(ext_mc_logits / ext_temp_used)
    ext_mc_mean = ext_mc_cal.mean(axis=0)
    ext_mc_std = ext_mc_cal.std(axis=0)
    ext_mc_entropy = entropy_binary(ext_mc_mean)

    ext_cal_resid = calibration_residual(ext_mc_mean, ext_val_bins)

    ext_unc_df = pd.DataFrame({
        'dataset': 'Chest_Xray_external',
        'y_true': ext_y_test,
        'p_raw': ext_p_test_raw,
        'p_calibrated': ext_p_test_cal,
        'p_mc_calibrated_mean': ext_mc_mean,
        'mc_std': ext_mc_std,
        'predictive_entropy': ext_mc_entropy,
        'calibration_residual_from_validation_bins': ext_cal_resid
    })
    ext_unc_path = TABLE_DIR / 'external_mc_dropout_uncertainty_calibrated.csv'
    ext_unc_df.to_csv(ext_unc_path, index=False)

    plt.figure(figsize=(6, 4))
    plt.hist(ext_mc_entropy, bins=25)
    plt.xlabel('Predictive entropy')
    plt.ylabel('Number of cases')
    plt.title('External Uncertainty Distribution')
    plt.tight_layout()
    ext_ent_fig = FIG_DIR / 'external_predictive_entropy_distribution.png'
    plt.savefig(ext_ent_fig, dpi=300, bbox_inches='tight')
    plt.show()

    log_message(f'Saved external calibrated MC-dropout uncertainty table to {ext_unc_path}')
    log_message(f'External mean predictive entropy: {ext_mc_entropy.mean():.4f}')
    log_message(f'External mean MC std: {ext_mc_std.mean():.4f}')






In [ ]:
# =========================================================
# 16.6 Apply MCDS layer to the external dataset
# =========================================================

if RUN_EXTERNAL_VALIDATION:
    # The original mcds_action_scores function stores y_true using the global y_test.
    # The labels are used for reporting only, never for action scoring.
    # We temporarily replace y_test so the exported external MCDS table carries
    # the correct ground-truth labels, then restore the original PneumoniaMNIST labels.
    _pneumoniamnist_y_test_backup = y_test.copy()
    y_test = ext_y_test.copy()
    try:
        ext_score_df = mcds_action_scores(
            ext_mc_mean,
            ext_mc_entropy,
            ext_mc_std,
            ext_cal_resid,
            SCENARIO
        )
    finally:
        y_test = _pneumoniamnist_y_test_backup

    ext_best_actions = ext_score_df.loc[
        ext_score_df.groupby('case_id')['mcds_score'].idxmin()
    ].reset_index(drop=True)

    ext_score_path = TABLE_DIR / 'external_mcds_action_scores.csv'
    ext_best_path = TABLE_DIR / 'external_mcds_selected_actions.csv'
    ext_score_df.to_csv(ext_score_path, index=False)
    ext_best_actions.to_csv(ext_best_path, index=False)

    ext_action_dist = ext_best_actions['action'].value_counts().rename_axis('action').reset_index(name='count')
    ext_action_dist['percentage'] = 100 * ext_action_dist['count'] / len(ext_best_actions)
    ext_action_dist['decision_entropy'] = -(ext_action_dist['percentage'] / 100) * np.log((ext_action_dist['percentage'] / 100).clip(1e-12, 1))
    ext_policy_entropy = float(ext_action_dist['decision_entropy'].sum())
    ext_policy_effective_actions = float(np.exp(ext_policy_entropy))

    ext_action_path = TABLE_DIR / 'external_mcds_action_distribution.csv'
    ext_action_dist.to_csv(ext_action_path, index=False)
    print(ext_action_dist[['action', 'count', 'percentage']])
    print(f'External policy entropy: {ext_policy_entropy:.4f}; effective actions: {ext_policy_effective_actions:.2f}')

    plt.figure(figsize=(9, 4.5))
    plt.bar(ext_action_dist['action'], ext_action_dist['count'])
    plt.xticks(rotation=30, ha='right')
    plt.ylabel('Number of cases')
    plt.title('External MCDS Workflow Action Distribution')
    plt.tight_layout()
    ext_action_fig = FIG_DIR / 'external_mcds_workflow_action_distribution.png'
    plt.savefig(ext_action_fig, dpi=300, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(7.5, 5.5))
    for action in ACTIONS:
        sub = ext_best_actions[ext_best_actions['action'] == action]
        if len(sub) > 0:
            plt.scatter(
                sub['pneumonia_probability_calibrated_mc'],
                sub['predictive_uncertainty_normalized'],
                s=24,
                alpha=0.75,
                label=action
            )
    plt.xlabel('Calibrated pneumonia probability')
    plt.ylabel('Normalized predictive uncertainty')
    plt.title('External MCDS Decision Landscape')
    plt.legend(fontsize=8, loc='best')
    plt.tight_layout()
    ext_landscape_fig = FIG_DIR / 'external_mcds_decision_landscape_probability_uncertainty.png'
    plt.savefig(ext_landscape_fig, dpi=300, bbox_inches='tight')
    plt.show()

    log_message(f'Saved external MCDS action scores to {ext_score_path}')
    log_message(f'Saved external selected actions to {ext_best_path}')
    log_message(f'Saved external MCDS action distribution to {ext_action_path}')
    log_message(f'External policy entropy: {ext_policy_entropy:.4f}; effective actions: {ext_policy_effective_actions:.2f}')






In [ ]:
# =========================================================
# 16.7 External safety audit and threshold comparison
# =========================================================

if RUN_EXTERNAL_VALIDATION:
    ext_audit = ext_best_actions.copy()
    ext_audit['is_pneumonia'] = ext_audit['y_true'].map({0: 'Normal', 1: 'Pneumonia'})
    ext_safety_table = pd.crosstab(ext_audit['action'], ext_audit['is_pneumonia'])
    ext_safety_path = TABLE_DIR / 'external_mcds_safety_audit_action_by_truth.csv'
    ext_safety_table.to_csv(ext_safety_path)
    print(ext_safety_table)
    log_message(f'Saved external MCDS safety audit table to {ext_safety_path}')

    ext_unsafe_mask = (
        (ext_audit['y_true'] == 1)
        & ext_audit['action'].str.contains('Low-risk monitoring', regex=False)
    )
    ext_unsafe_cases = ext_audit.loc[ext_unsafe_mask, [
        'case_id',
        'pneumonia_probability_calibrated_mc',
        'predictive_entropy',
        'predictive_uncertainty_normalized',
        'mc_std',
        'calibration_gap',
        'action',
        'mcds_score'
    ]]
    ext_unsafe_path = TABLE_DIR / 'external_potentially_unsafe_low_action_pneumonia_cases.csv'
    ext_unsafe_cases.to_csv(ext_unsafe_path, index=False)

    ext_high_prob_pneumonia = ext_audit[
        (ext_audit['y_true'] == 1)
        & (ext_audit['pneumonia_probability_calibrated_mc'] >= 0.70)
    ]
    ext_high_prob_low_action = ext_high_prob_pneumonia[
        ext_high_prob_pneumonia['action'].str.contains('Low-risk monitoring', regex=False)
    ]

    print(f'External potentially unsafe low-action pneumonia cases: {len(ext_unsafe_cases)}')
    print(f'External high-probability pneumonia cases assigned to low monitoring: {len(ext_high_prob_low_action)}')
    log_message(f'External potentially unsafe low-action pneumonia cases: {len(ext_unsafe_cases)}')
    log_message(f'External high-probability pneumonia cases assigned to low monitoring: {len(ext_high_prob_low_action)}')

    # Threshold comparison on external dataset.
    ext_sens_rows = []
    for t in np.linspace(0.1, 0.9, 17):
        pred = (ext_mc_mean >= t).astype(int)
        cm = confusion_matrix(ext_y_test, pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        ext_sens_rows.append({
            'threshold': float(t),
            'accuracy': accuracy_score(ext_y_test, pred),
            'balanced_accuracy': balanced_accuracy_score(ext_y_test, pred),
            'precision': precision_score(ext_y_test, pred, zero_division=0),
            'recall_sensitivity': recall_score(ext_y_test, pred, zero_division=0),
            'specificity': tn / (tn + fp) if (tn + fp) > 0 else np.nan,
            'false_negatives': int(fn),
            'false_positives': int(fp),
            'tp': int(tp),
            'tn': int(tn)
        })

    ext_sens_df = pd.DataFrame(ext_sens_rows)
    ext_sens_path = TABLE_DIR / 'external_probability_threshold_sensitivity_vs_mcds.csv'
    ext_sens_df.to_csv(ext_sens_path, index=False)
    print(ext_sens_df.round(4))
    log_message(f'Saved external threshold sensitivity analysis to {ext_sens_path}')

    plt.figure(figsize=(6, 4))
    plt.plot(ext_sens_df['threshold'], ext_sens_df['recall_sensitivity'], marker='o', label='Sensitivity')
    plt.plot(ext_sens_df['threshold'], ext_sens_df['specificity'], marker='o', label='Specificity')
    plt.plot(ext_sens_df['threshold'], ext_sens_df['balanced_accuracy'], marker='o', label='Balanced accuracy')
    plt.xlabel('Probability threshold')
    plt.ylabel('Metric')
    plt.title('External Threshold Sensitivity')
    plt.legend()
    plt.tight_layout()
    ext_th_fig = FIG_DIR / 'external_probability_threshold_sensitivity.png'
    plt.savefig(ext_th_fig, dpi=300, bbox_inches='tight')
    plt.show()
    log_message(f'Saved external threshold sensitivity figure to {ext_th_fig}')






## External fine-tuning and domain-shift characterization

This optional extension compares the frozen external transfer model with a lightly fine-tuned condition. It is disabled by default because it requires additional GPU time.






In [ ]:
# =========================================================
# Optional external fine-tuning condition
# =========================================================

RUN_EXTERNAL_FINETUNE = False
if RUN_EXTERNAL_VALIDATION and RUN_EXTERNAL_FINETUNE:
    log_message('Starting external fine-tuning condition.')
    fine_model = build_external_transfer_model()
    for layer in fine_model.layers:
        if hasattr(layer, 'layers') and len(layer.layers) > 30:
            layer.trainable = True
            for sub_layer in layer.layers[:-30]:
                sub_layer.trainable = False
            for sub_layer in layer.layers[-30:]:
                sub_layer.trainable = True
    fine_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy', keras.metrics.AUC(name='auc')])
    fine_model.fit(ext_train_ds, validation_data=ext_val_ds, epochs=10,
                   callbacks=[keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=4, restore_best_weights=True)],
                   verbose=1)
    fine_raw = fine_model.predict(ext_test_ds, verbose=0).reshape(-1)
    fine_tune_comparison = pd.DataFrame([
        {'condition': 'frozen_backbone', 'roc_auc': ext_cal_metrics['roc_auc'], 'brier_score': ext_cal_metrics['brier_score'], 'balanced_accuracy': ext_cal_metrics['balanced_accuracy']},
        {'condition': 'fine_tuned', 'roc_auc': roc_auc_score(ext_y_test, fine_raw), 'brier_score': brier_score_loss(ext_y_test, fine_raw), 'balanced_accuracy': balanced_accuracy_score(ext_y_test, (fine_raw >= 0.5).astype(int))}
    ])
else:
    fine_tune_comparison = pd.DataFrame([{'condition': 'fine_tuning_not_run', 'note': 'Set RUN_EXTERNAL_FINETUNE=True to run this optional condition.'}])
fine_tune_path = TABLE_DIR / 'external_frozen_vs_finetuned_comparison.csv'
fine_tune_comparison.to_csv(fine_tune_path, index=False)
print(fine_tune_comparison)
log_message(f'Saved external frozen-vs-finetuned comparison to {fine_tune_path}')






## External MCDS ablation

This cell applies the same ablation logic to the second chest X-ray dataset when available. It tests whether the structured evaluation behavior remains meaningful under a distinct imaging distribution.





In [ ]:
# =========================================================
# External MCDS ablation study
# =========================================================

if RUN_EXTERNAL_VALIDATION:
    ext_ablation_rows = []
    ext_ablation_rows.append(summarize_policy(ext_best_actions['action'], ext_y_test,
                                              prob=ext_best_actions['pneumonia_probability_calibrated_mc'],
                                              label='external_full_mcds'))
    for variant in ['no_safety_barrier', 'no_feasibility_gate', 'no_profile_distance', 'no_confidence_reward']:
        ext_selected_variant = select_from_score_variant(ext_score_df, variant)
        ext_selected_variant.to_csv(TABLE_DIR / f'external_mcds_selected_actions_{variant}.csv', index=False)
        ext_ablation_rows.append(summarize_policy(ext_selected_variant['action'], ext_y_test,
                                                  prob=ext_selected_variant['pneumonia_probability_calibrated_mc'],
                                                  label=f'external_{variant}'))
    ext_plain_actions = np.where(ext_mc_mean >= EXT_CAL_OPT_THR,
                                 'Treat as likely pneumonia with clinician confirmation',
                                 'Low-risk monitoring / outpatient follow-up')
    ext_ablation_rows.append(summarize_policy(ext_plain_actions, ext_y_test, prob=ext_mc_mean,
                                              label='external_plain_threshold_classifier'))
    ext_ablation_df = pd.DataFrame(ext_ablation_rows)
    ext_ablation_path = TABLE_DIR / 'external_mcds_ablation_comparison.csv'
    ext_ablation_df.to_csv(ext_ablation_path, index=False)
    print(ext_ablation_df.round(4))
    log_message(f'Saved external MCDS ablation comparison to {ext_ablation_path}')
else:
    print('External validation was skipped; external ablation was not run.')







In [ ]:
# =========================================================
# 16.8 External validation manuscript-oriented summary
# =========================================================

if RUN_EXTERNAL_VALIDATION:
    ext_summary_lines = []
    ext_summary_lines.append('External Chest-Xray Transferability Validation')
    ext_summary_lines.append('=' * 72)
    ext_summary_lines.append(f'Dataset path: {EXTERNAL_DATASET_PATH}')
    ext_summary_lines.append(f'Image size: {EXT_IMG_SIZE} x {EXT_IMG_SIZE}')
    ext_summary_lines.append(f'Raw ROC-AUC: {ext_raw_metrics["roc_auc"]:.4f}')
    ext_summary_lines.append(f'Raw balanced accuracy: {ext_raw_metrics["balanced_accuracy"]:.4f}')
    ext_summary_lines.append(f'Calibrated ROC-AUC: {ext_cal_metrics["roc_auc"]:.4f}')
    ext_summary_lines.append(f'Calibrated balanced accuracy: {ext_cal_metrics["balanced_accuracy"]:.4f}')
    ext_summary_lines.append(f'External optimized temperature: {EXT_TEMP:.4f}')
    ext_summary_lines.append(f'External validation ECE: {ext_val_ece:.4f}')
    ext_summary_lines.append(f'External test ECE: {ext_test_ece:.4f}')
    ext_summary_lines.append(f'External mean MC-dropout entropy: {ext_mc_entropy.mean():.4f}')
    ext_summary_lines.append(f'External mean MC-dropout std: {ext_mc_std.mean():.4f}')
    ext_summary_lines.append(f'External MCDS policy entropy: {ext_policy_entropy:.4f}')
    ext_summary_lines.append(f'External effective number of actions: {ext_policy_effective_actions:.2f}')
    ext_summary_lines.append(f'External potentially unsafe low-action pneumonia cases: {len(ext_unsafe_cases)}')
    ext_summary_lines.append('')
    ext_summary_lines.append('Interpretation:')
    ext_summary_lines.append('This reduced second-dataset experiment evaluates transferability rather than full benchmark optimization. A frozen pretrained backbone supplies pneumonia evidence on a higher-resolution chest X-ray dataset, while the same calibration, MC-dropout uncertainty, and constrained MCDS structured evaluation layer are applied without redesigning the decision framework.')
    ext_summary_lines.append('')
    ext_summary_lines.append('Recommended manuscript framing:')
    ext_summary_lines.append('The external experiment should be presented as an additional transferability validation showing that the constrained MCDS evaluation layer can operate beyond PneumoniaMNIST and can still produce uncertainty-aware, safety-audited, multi-action recommendations under a distinct imaging distribution.')
    ext_summary_lines.append('')
    ext_summary_lines.append('External MCDS action distribution:')
    for _, row in ext_action_dist.iterrows():
        ext_summary_lines.append(f'- {row["action"]}: {int(row["count"])} cases ({row["percentage"]:.2f}%)')

    ext_summary_text = '\n'.join(ext_summary_lines)
    print(ext_summary_text)

    ext_interpret_path = OUTPUT_DIR / 'external_transferability_interpretation_summary.txt'
    with open(ext_interpret_path, 'w', encoding='utf-8') as f:
        f.write(ext_summary_text)

    with open(SUMMARY_PATH, 'a', encoding='utf-8') as f:
        f.write('\n\n' + ext_summary_text + '\n')

    log_message(f'Saved external transferability interpretation summary to {ext_interpret_path}')
else:
    print('External validation was skipped because the dataset path was not found.')






